# Ya-Risk — V26 SCHEMA-FIRST — Audit de TOUTES les formules + alertes par police rouge

Cette version conserve l'architecture V25 (ACTIF/PASSIF/TCR, TCR 1 ou 2 pages, schéma fermé, identité page par page) et renforce le contrôle Excel :

- **toutes les cellules de formule financières ACTIF / PASSIF / TCR qui correspondent à un champ de la liasse sont comparées à la valeur observée par Qwen**, et pas uniquement les totaux ;
- en cas d'écart ≥ 1 000 DZD, la cellule formule conserve son fond d'origine : **aucun remplissage/couleur de cellule n'est modifié** ;
- le **chiffre de la cellule passe en rouge** et un **commentaire Excel** détaille valeur PDF/Qwen, valeur calculée, écart DZD, niveau LOW/MEDIUM/HIGH, formule et source PDF/page ;
- les seuils restent : LOW ≥ 1 000 DZD, MEDIUM ≥ 10 000 DZD, HIGH ≥ 100 000 DZD ;
- les formules techniques du modèle sans contrepartie imprimée dans la liasse (ex. contrôles Actif = Passif en lignes 29/30 du PASSIF) ne sont pas comparées à Qwen ; elles restent des contrôles internes du modèle.

Couverture financière :
- **ACTIF** : agrégats, sous-totaux, total général et toutes les formules `Brut - amortissements/provisions = Net` présentes dans le modèle ;
- **PASSIF** : capitaux propres, passifs non courants, passifs courants, totaux I/II/III et total passif ;
- **TCR** : chaque formule Débit et Crédit des lignes calculées, séparément pour N et N-1, jusqu'au résultat net.

Les alertes sont appliquées uniquement après le dernier recalcul LibreOffice afin qu'elles persistent dans le fichier final.


In [ ]:
# CELLULE 1 — DEPENDANCES LEGERES
# Ne pas réinstaller torch/vLLM : ils proviennent de l'environnement GPU Domino.
%pip install -q -U "transformers>=4.57.0" accelerate pymupdf pillow psutil openpyxl
print("✅ Dépendances Python légères OK")


In [ ]:
# CELLULE 2 — IMPORTS V26
import os, re, json, time, gc, copy, shutil, subprocess, sys, unicodedata, traceback, tempfile
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import fitz
try:
    fitz.TOOLS.mupdf_display_errors(False)
    fitz.TOOLS.mupdf_display_warnings(False)
except Exception:
    pass
import torch
from PIL import Image
import openpyxl
from openpyxl import load_workbook
from openpyxl.comments import Comment
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side

print("✅ Imports OK")


In [ ]:
# CELLULE 3 — CONFIGURATION V26
MODEL_NAME = os.environ.get("QWEN_MODEL_NAME", "Qwen3.8-27B-FP8")
MODEL_PATH = os.environ.get(
    "QWEN_MODEL_PATH",
    "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B-FP8/main"
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

INPUT_DIR = Path("/mnt/Risk/bilans_in")
OUTPUT_DIR = Path("/mnt/Risk/bilans_out/GT/V26")
JSON_DIR = OUTPUT_DIR / "json_bruts_v26"
CONTROL_DIR = OUTPUT_DIR / "json_avant_certification_v26"
XLSX_DIR = OUTPUT_DIR / "excel_pre_certification_v26"
LOG_PATH = OUTPUT_DIR / "pipeline_v26.log"
INDEX_PATH = OUTPUT_DIR / "index_v26.jsonl"
for p in (OUTPUT_DIR, JSON_DIR, CONTROL_DIR, XLSX_DIR): p.mkdir(parents=True, exist_ok=True)

# Extraction active aujourd'hui. Les annexes sont classées et décrites par le schéma, sans extraction détaillée.
ACTIVE_TABLES = {"ACTIF", "PASSIF", "TCR"}
SCHEMA_VERSION = "26.0"
SCHEMA_REGISTRY_VERSION = "SERIE_G2_2011_SCHEMA_FIRST_V4_ALL_FORMULAS_REDFONT"
FORCE_REEXTRACT = False

# Réutilisation des extractions V25 : aucun besoin de refaire Qwen pour générer les Excel V26.
LEGACY_JSON_DIRS = [Path("/mnt/Risk/bilans_out/GT/V25/json_bruts_v25")]


# Contrôle identité client. Le dernier exercice est la référence.
IDENTITY_SIMILARITY_MIN = 0.92
IDENTITY_HEADER_FRACTION = 0.30
IDENTITY_HEADER_ZOOM = 1.80
IDENTITY_HEADER_MAX_SIDE = 1000
IDENTITY_BATCH_SIZE = 16

# Profil vitesse / image (V23 avait validé ce principe sur votre H100).
ROUTER_MAX_SIDE = 768
EXTRACT_MAX_SIDE = 1536
RETRY_MAX_SIDE = 2024
PDF_ROUTER_ZOOM = 1.15
PDF_EXTRACT_ZOOM = 2.25
BLANK_THRESHOLD = 0.985
CLASSIF_BATCH_SIZE = 16
GPU_BATCH_SIZE = 8
PDF_WORKERS = 4

# Retry uniquement si le résultat est anormalement pauvre — ce seuil ne décide jamais du type de page.
MIN_VALUES_RETRY = {"ACTIF":10, "PASSIF":8, "TCR_P1":8, "TCR_P2":4, "TCR_FULL":12}

# Politique d'écarts demandée — DZD absolus.
ALERT_MIN_DZD = 1_000.0
ALERT_MEDIUM_DZD = 10_000.0
ALERT_HIGH_DZD = 100_000.0

def ecart_level(ecart_dzd):
    a = abs(float(ecart_dzd or 0.0))
    if a < ALERT_MIN_DZD: return "OK"
    if a < ALERT_MEDIUM_DZD: return "LOW"
    if a < ALERT_HIGH_DZD: return "MEDIUM"
    return "HIGH"

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

print("Modele:", MODEL_NAME)
print("Device:", DEVICE)
print("PDF détectés:", len(list(INPUT_DIR.rglob("*.pdf"))) if INPUT_DIR.exists() else 0)
print("Seuils écarts DZD:", ALERT_MIN_DZD, ALERT_MEDIUM_DZD, ALERT_HIGH_DZD)
print("Seuil similarité raison sociale:", f"{IDENTITY_SIMILARITY_MIN:.0%}")


In [ ]:
# CELLULE 4 — LOG + OUTILS GENERAUX + IDENTITE
def log(msg):
    line = datetime.now().strftime("%Y-%m-%d %H:%M:%S") + " — " + str(msg)
    print(line)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(LOG_PATH, "a", encoding="utf-8") as f: f.write(line + "\n")

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", str(s or "")) if not unicodedata.combining(c))

def norm_text(s):
    return re.sub(r"[^a-z0-9]+", " ", strip_accents(s).lower()).strip()

def norm_nif(v):
    d = re.sub(r"\D", "", str(v or ""))
    return d if len(d) == 15 else None

def norm_amount(v):
    if v is None or isinstance(v, bool): return None
    if isinstance(v, (int,float)): return float(v)
    s = str(v).strip().replace(" ", " ")
    if not s or s.lower() in {"null","none","-","—",""}: return None
    neg = s.startswith("(") and s.endswith(")")
    s = s.strip("()").replace(" ", "").replace("DZD", "").replace("DA", "")
    # Les liasses algériennes utilisent fréquemment l'espace comme séparateur de milliers.
    # Si une virgule est le seul séparateur décimal, elle est convertie en point.
    if "," in s and "." not in s and len(s.rsplit(",",1)[-1]) <= 2:
        s = s.replace(",", ".")
    else:
        s = s.replace(",", "")
    s = re.sub(r"[^0-9.\-+]", "", s)
    try:
        x = float(s)
        return -x if neg else x
    except Exception:
        return None

def parse_year(v):
    if v is None: return None
    s = str(v)
    ys = re.findall(r"(?<!\d)(20\d{2}|19\d{2})(?!\d)", s)
    if ys: return int(ys[-1])
    # dates abrégées jj/mm/23
    m = re.search(r"(?:^|\D)(\d{2})\s*$", s)
    if m:
        yy=int(m.group(1)); return 2000+yy if yy < 70 else 1900+yy
    return None

def chunks(seq, n):
    for i in range(0, len(seq), n): yield seq[i:i+n]


def norm_company_name(v):
    # Neutralise casse/accents/ponctuation et recolle les acronymes écrits lettre par lettre
    # (S.P.A = SPA, B.N.P = BNP) sans supprimer les formes juridiques.
    base=re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9]+", " ", strip_accents(str(v or "")).upper())).strip()
    toks=base.split(); out=[]; run=[]
    def flush():
        nonlocal run
        if run: out.append("".join(run) if len(run)>=2 else run[0]); run=[]
    for t in toks:
        if len(t)==1 and t.isalpha(): run.append(t)
        else: flush(); out.append(t)
    flush()
    return " ".join(out)

def company_similarity(a,b):
    aa,bb=norm_company_name(a),norm_company_name(b)
    if not aa or not bb: return None
    return SequenceMatcher(None,aa,bb,autojunk=False).ratio()

def normalize_page_identity(ent):
    ent=dict(ent or {})
    raw_nif=ent.get("nif_15") or ent.get("nif")
    return {
        "entreprise": (str(ent.get("entreprise")).strip() if ent.get("entreprise") not in (None,"") else None),
        "nif": (str(raw_nif).strip() if raw_nif not in (None,"") else None),
        "nif_15": norm_nif(raw_nif),
        "exercice": ent.get("exercice"),
        "exercice_du": ent.get("exercice_du"),
        "exercice_au": ent.get("exercice_au"),
        "identity_source": ent.get("identity_source"),
    }

def merge_identity(primary, fallback):
    p=normalize_page_identity(primary); f=normalize_page_identity(fallback)
    out={}
    for k in ["entreprise","nif","nif_15","exercice","exercice_du","exercice_au","identity_source"]:
        out[k]=p.get(k) if p.get(k) not in (None,"") else f.get(k)
    return out


In [ ]:
# CELLULE 5 — REFERENTIEL OFFICIEL SERIE G N°2 + FIELD_ID UNIQUES
TABLE_SCHEMAS = {'ACTIF': {'titre': 'BILAN (ACTIF)',
           'cols': ['montant_brut', 'amortissements_provisions_pertes', 'net_n', 'net_n1'],
           'postes': {'ecarts_acquisition_goodwill': 'Ecart d acquisition goodwill',
                      'immobilisations_incorporelles': 'Immobilisations incorporelles',
                      'terrains': 'Terrains',
                      'batiments': 'Batiments',
                      'autres_immobilisations_corporelles': 'Autres Immobilisations corporelles',
                      'immobilisations_corporelles': 'Immobilisations corporelles',
                      'immobilisations_en_concession': 'Immobilisations en concession',
                      'immobilisations_en_cours': 'Immobilisations en cours',
                      'titres_mis_en_equivalence': 'Titres mis en equivalence',
                      'autres_participations_creances': 'Autres participations et creances rattachees',
                      'autres_titres_immobilises': 'Autres titres immobilises',
                      'prets_actifs_financiers_non_courants': 'Prets et autres actifs financiers non courants',
                      'impots_differes_actif': 'Impots Differes Actif',
                      'immobilisations_financieres': 'Immobilisations financieres',
                      'total_actif_non_courant': 'TOTAL ACTIF NON COURANT',
                      'stocks_encours': 'Stocks et encours',
                      'clients': 'Clients',
                      'autres_debiteurs': 'Autres debiteurs',
                      'impots_assimiles_actif': 'Impots et assimiles',
                      'autres_creances_assimiles': 'Autres Creances et Emplois assimiles',
                      'creances_emplois_assimiles': 'Creances et emplois assimiles',
                      'disponibilites_assimiles': 'Disponibilites et assimiles',
                      'placements_financiers_courants': 'Placements et autres actifs financiers courants',
                      'tresorerie_actif': 'Tresorerie',
                      'total_actif_courant': 'TOTAL ACTIF COURANT',
                      'total_general_actif': 'TOTAL GENERAL ACTIF'},
           'table_id': 'ACTIF'},
 'PASSIF': {'titre': 'BILAN (PASSIF)',
            'cols': ['n', 'n1'],
            'postes': {'capital_emis': 'Capital emis',
                       'capital_non_appele': 'Capital non appele',
                       'primes_reserves': 'Primes et reserves',
                       'ecart_reevaluation': 'Ecart de reevaluation',
                       'ecart_equivalence': 'Ecart d equivalence',
                       'resultat_net_passif': 'Resultat net',
                       'report_a_nouveau': 'Report a nouveau',
                       'part_societe_consolidante': 'Part de la societe consolidante',
                       'part_minoritaires': 'Part des minoritaires',
                       'total_capitaux_propres': 'TOTAL I',
                       'emprunts_dettes_financieres': 'Emprunts et dettes financieres',
                       'impots_differes_provisionnes': 'Impots differes et provisionnes',
                       'autres_dettes_non_courantes': 'Autres dettes non courantes',
                       'provisions_produits_avance': 'Provisions et produits constatés d avance',
                       'total_passifs_non_courants': 'TOTAL PASSIFS NON COURANTS II',
                       'fournisseurs_rattaches': 'Fournisseurs et comptes rattaches',
                       'impots_passif': 'Impots',
                       'autres_dettes': 'Autres dettes',
                       'tresorerie_passif': 'Tresorerie Passif',
                       'total_passifs_courants': 'TOTAL PASSIFS COURANTS',
                       'total_general_passif': 'TOTAL GENERAL PASSIF'},
            'table_id': 'PASSIF'},
 'TCR': {'titre': 'COMPTE DE RESULTAT',
         'cols': ['n_debit', 'n_credit', 'n1_debit', 'n1_credit'],
         'postes': {'ventes_marchandises': 'Ventes de Marchandises',
                    'produits_fabriques': 'Produits Fabriques',
                    'prestations_services': 'Prestations de Services',
                    'ventes_travaux': 'Ventes de Travaux',
                    'produits_annexes': 'Produits Annexes',
                    'rabais_remises_ristournes_accordes': 'Rabais remises ristournes accordes',
                    'chiffre_affaires_net': 'Chiffre d affaires net',
                    'production_stockee_destockee': 'Production Stockee ou destockee',
                    'production_immobilisee': 'Production immobilisee',
                    'subvention_exploitation': 'Subvention d exploitation',
                    'production_exercice': 'I-Production de l exercice',
                    'achats_marchandises_vendues': 'Achats de Marchandises vendues',
                    'matieres_premieres': 'Matieres premieres',
                    'autres_approvisionnements': 'Autres Approvisionnements',
                    'variation_stocks': 'Variation des Stocks',
                    'achats_etudes_prestations': 'Achats d Etudes et de Prestations de services',
                    'rabais_remises_ristournes_obtenus_achats': 'Rabais remises ristournes obtenus sur achats',
                    'autres_consommations': 'Autres consommations',
                    'sous_traitance_generale': 'Sous-traitance generale',
                    'locations': 'Locations',
                    'entretien_reparations': 'Entretien reparations et maintenance',
                    'primes_assurances': 'Primes d assurances',
                    'personnel_exterieur': 'Personnel exterieur a l entreprise',
                    'remuneration_intermediaires': 'Remuneration d intermediaires et honoraires',
                    'publicite': 'Publicite',
                    'deplacements_missions': 'Deplacements missions et receptions',
                    'rabais_remises_ristournes_obtenus_services': 'Rabais remises ristournes obtenus sur services exterieurs',
                    'autres_services': 'Autres services',
                    'consommations_exercice': 'II-Consommations de l exercice',
                    'valeur_ajoutee_exploitation': 'III-Valeur ajoutee d exploitation',
                    'charges_personnel': 'Charges de personnel',
                    'impots_taxes_assimiles': 'Impots et taxes et versements assimiles',
                    'excedent_brut_exploitation': 'IV-Excedent brut d exploitation',
                    'autres_produits_operationnels': 'Autres produits operationnels',
                    'autres_charges_operationnelles': 'Autres charges operationnelles',
                    'dotations_amortissements': 'Dotations aux amortissements',
                    'provisions': 'Provisions',
                    'pertes_valeur': 'Perte de Valeur',
                    'reprises_pertes_valeur_provisions': 'Reprise sur pertes de valeur et provisions',
                    'resultat_operationnel': 'V-Resultat operationnel',
                    'produits_financiers': 'Produits financiers',
                    'charges_financieres': 'Charges financieres',
                    'resultat_financier': 'VI-Resultat Financier',
                    'resultat_ordinaire': 'VII-Resultat ordinaire',
                    'elements_extraordinaires_produits': 'Elements extraordinaires Produits',
                    'elements_extraordinaires_charges': 'Elements extraordinaires Charges',
                    'resultat_extraordinaire': 'VIII-Resultat extraordinaire',
                    'impots_exigibles_resultats': 'Impots exigibles sur resultats',
                    'impots_differes_resultats': 'Impots differes sur resultats',
                    'resultat_net_exercice': 'RESULTAT NET DE L EXERCICE'},
         'table_id': 'TCR'},
 'A01': {'titre': '1/ Tableau des mouvements des stocks',
         'cols': ['solde_debut', 'debit', 'credit', 'solde_fin'],
         'postes': {'stocks_marchandises': 'Stocks de marchandises',
                    'matieres_fournitures': 'Matieres et fournitures',
                    'autres_approvisionnements': 'Autres approvisionnements',
                    'encours_production_biens': 'Encours de production de biens',
                    'encours_production_services': 'Encours de production de services',
                    'stocks_produits': 'Stocks de produits',
                    'stocks_provenant_immobilisations': 'Stocks provenant d immobilisations',
                    'stocks_exterieur': 'Stocks a l exterieur',
                    'total': 'TOTAL'},
         'table_id': 'A01'},
 'A03': {'titre': '3/ Charges de personnel, impots, taxes, autres services',
         'cols': ['montant'],
         'postes': {'charges_locatives': 'Charges locatives et charges de copropriete',
                    'etudes_recherches': 'Etudes et recherches',
                    'documentation_divers': 'Documentation et divers',
                    'transports_biens': 'Transports de biens et transport collectif du personnel',
                    'frais_postaux': 'Frais postaux et de telecommunications',
                    'services_bancaires': 'Services bancaires et assimiles',
                    'cotisations_divers': 'Cotisations et divers',
                    'total_autres_services': 'TOTAL (1)',
                    'remunerations_personnel': 'Remunerations du personnel',
                    'remuneration_exploitant': 'Remunerations de l exploitant individuel (cas d une EURL)',
                    'cotisations_sociales': 'Cotisations aux organismes sociaux',
                    'charges_sociales_exploitant': 'Charges sociales de l exploitant individuel (cas d une EURL)',
                    'autres_charges_sociales': 'Autres charges sociales',
                    'autres_charges_personnel': 'Autres charges de personnels',
                    'total_charges_personnel': 'TOTAL (2)',
                    'impots_sur_remunerations': 'Impots taxes et versements assimiles sur remunerations',
                    'impots_non_recuperables': 'Impots et taxes non recuperables sur chiffres d affaires',
                    'autres_impots_taxes': 'Autres impots et taxes (hors impots sur les resultats)',
                    'total_impots': 'TOTAL (3)',
                    'total_general': 'TOTAL (1)+(2)+(3)'},
         'table_id': 'A03'},
 'A04': {'titre': '4/ Autres charges et produits operationnels',
         'cols': ['montant'],
         'postes': {'redevances_concessions_charges': 'Redevances pour concessions brevets licences logiciels (charges)',
                    'moins_values_sorties_actifs': 'Moins values sur sorties d actifs immobilises non financiers',
                    'jetons_presence_charges': 'Jetons de presence (charges)',
                    'pertes_creances_irrecouvrables': 'Perte sur creances irrecouvrables',
                    'quote_part_operations_commun_charges': 'Quote-part de resultat sur operations faites en commun (charges)',
                    'amendes_penalites_dons': 'Amendes et penalites subventions accordees dons et liberalites',
                    'charges_exceptionnelles_gestion': 'Charges exceptionnelles de gestion courante',
                    'autres_charges_gestion': 'Autres charges de gestion courante',
                    'total_charges': 'TOTAL (charges)',
                    'redevances_concessions_produits': 'Redevances pour concessions brevets licences logiciels (produits)',
                    'plus_values_sorties_actifs': 'Plus values sur sorties d actifs immobilises non financiers',
                    'jetons_presence_produits': 'Jetons de presence et remunerations d administrateurs ou de gerant',
                    'quotes_parts_subventions_virees': 'Quotes-parts de subventions d investissement virees au resultat',
                    'quote_part_operations_commun_produits': 'Quote-part de resultat sur operations faites en commun (produits)',
                    'rentrees_creances_amorties': 'Rentree sur creances amorties',
                    'produits_exceptionnels_gestion': 'Produits exceptionnels sur operations de gestion',
                    'autres_produits_gestion': 'Autres produits de gestion courante',
                    'total_produits': 'TOTAL (produits)'},
         'table_id': 'A04'},
 'A05': {'titre': '5/ Tableau des amortissements et pertes de valeurs',
         'cols': ['dotations_cumulees_debut',
                  'dotations_exercice',
                  'diminutions_elements_sortis',
                  'dotations_cumulees_fin',
                  'dotations_fiscales_exercice',
                  'ecarts'],
         'postes': {'goodwill': 'Goodwill',
                    'immobilisations_incorporelles': 'Immobilisations incorporelles',
                    'immobilisations_corporelles': 'Immobilisations corporelles',
                    'participations': 'Participations',
                    'autres_actifs_financiers_non_courants': 'Autres actifs financiers non courants',
                    'total': 'TOTAL'},
         'table_id': 'A05'},
 'A06': {'titre': '6/ Tableau des immobilisations creees ou acquises',
         'cols': ['montants_bruts', 'tva_deduite', 'montant_net_a_amortir'],
         'postes': {'goodwill': 'Goodwill',
                    'immobilisations_incorporelles': 'Immobilisations incorporelles',
                    'immobilisations_corporelles': 'Immobilisations corporelles',
                    'participations': 'Participations',
                    'autres_actifs_financiers_non_courants': 'Autres actifs financiers non courants',
                    'total': 'TOTAL'},
         'table_id': 'A06'},
 'A08': {'titre': '8/ Tableau des provisions et pertes de valeurs',
         'cols': ['provisions_cumulees_debut', 'dotations_exercice', 'reprises_exercice', 'provisions_cumulees_fin'],
         'postes': {'pertes_valeur_stocks': 'Pertes de valeurs sur stocks',
                    'pertes_valeur_creances': 'Pertes de valeurs sur creances',
                    'pertes_valeur_actions': 'Pertes de valeurs sur actions et parts sociales',
                    'provisions_pensions': 'Provisions pour pensions et obligations similaires',
                    'provisions_litiges': 'Provisions sur litiges',
                    'autres_provisions_personnel': 'Autres provisions liees au personnel',
                    'provisions_impots': 'Provisions pour impots',
                    'autres_provisions': 'Autres provisions',
                    'total': 'TOTAL'},
         'table_id': 'A08'},
 'A09': {'titre': '9/ Tableau de determination du resultat fiscal',
         'cols': ['montant'],
         'postes': {'resultat_net_benefice': 'I. Resultat net de l exercice Benefice',
                    'resultat_net_perte': 'I. Resultat net de l exercice Perte',
                    'charges_immeubles_non_affectes': 'Charges des immeubles non affectes directement a l exploitation',
                    'quote_part_cadeaux_publicitaires': 'Quote-part des cadeaux publicitaires non deductibles',
                    'quote_part_sponsoring': 'Quote-part du sponsoring et parrainage non deductibles',
                    'frais_reception': 'Frais de reception non deductibles',
                    'cotisations_dons': 'Cotisations et dons non deductibles',
                    'impots_taxes_non_deductibles': 'Impots et taxes non deductibles',
                    'provisions_non_deductibles': 'Provisions non deductibles',
                    'amortissements_non_deductibles': 'Amortissements non deductibles',
                    'quote_part_frais_rd': 'Quote-part des frais de recherche developpement non deductibles',
                    'amortissements_credit_bail_preneur': 'Amortissements non deductibles credit bail (Preneur)',
                    'loyers_hors_produits_financiers_bailleur': 'Loyers hors produits financiers (bailleur)',
                    'ibs_impot_exigible': 'Impots sur les benefices - Impot exigible sur le resultat',
                    'ibs_impot_differe': 'Impots sur les benefices - Impot differe (variation)',
                    'pertes_valeur_non_deductibles': 'Pertes de valeurs non deductibles',
                    'amendes_penalites': 'Amendes et penalites',
                    'autres_reintegrations': 'Autres reintegrations',
                    'total_reintegrations': 'Total des reintegrations',
                    'plus_values_cession_actif_immobilise': 'Plus values sur cession d elements d actif immobilises',
                    'produits_plus_values_actions_bourse': 'Produits et plus values de cession des actions et titres assimiles cotes en bourse',
                    'revenus_distribution_benefices': 'Revenus provenant de la distribution des benefices',
                    'amortissements_credit_bail_bailleur': 'Amortissements credit bail (Bailleur)',
                    'loyers_hors_charges_financieres_preneur': 'Loyers hors charges financieres (Preneur)',
                    'complement_amortissements': 'Complement d amortissements',
                    'autres_deductions': 'Autres deductions',
                    'total_deductions': 'Total des deductions',
                    'total_deficits_a_deduire': 'Total des deficits a deduire',
                    'resultat_fiscal_benefice': 'Resultat fiscal (I+II-III-IV) Benefice',
                    'resultat_fiscal_deficit': 'Resultat fiscal (I+II-III-IV) Deficit',
                    'deficit_anterieur_1': 'Déficit antérieur 1 (année à lire sur le formulaire)',
                    'deficit_anterieur_2': 'Déficit antérieur 2 (année à lire sur le formulaire)',
                    'deficit_anterieur_3': 'Déficit antérieur 3 (année à lire sur le formulaire)',
                    'deficit_anterieur_4': 'Déficit antérieur 4 (année à lire sur le formulaire)'},
         'table_id': 'A09'},
 'A10': {'titre': '10/ Tableau d affectation du resultat et des reserves (N-1)',
         'cols': ['montant'],
         'postes': {'origine_report_a_nouveau_n1': 'Report a nouveau de l exercice N-1',
                    'origine_resultat_n1': 'Resultat de l exercice N-1',
                    'origine_prelevements_reserves': 'Prelevements sur reserves',
                    'origine_total': 'TOTAL (origine)',
                    'affectation_reserves': 'Reserves',
                    'affectation_augmentation_capital': 'Augmentation du capital',
                    'affectation_dividendes': 'Dividendes',
                    'affectation_report_a_nouveau': 'Report a nouveau',
                    'affectation_total': 'TOTAL (affectation)'},
         'table_id': 'A10'},
 'A02': {'table_id': 'A02',
         'titre': '2/ Tableau de la fluctuation de la production stockée',
         'dynamic': True,
         'cols': ['debit', 'credit', 'solde_debiteur', 'solde_crediteur'],
         'str_cols': [],
         'postes': {}},
 'A07': {'table_id': 'A07',
         'titre': '7/ Tableau des immobilisations cédées (plus ou moins value) au cours de l’exercice',
         'dynamic': True,
         'cols': ['nature_immobilisation_cedee',
                  'date_acquisition',
                  'montant_net_actif',
                  'amortissements_pratiques',
                  'valeur_nette_comptable',
                  'prix_cession',
                  'plus_value',
                  'moins_value'],
         'str_cols': ['nature_immobilisation_cedee', 'date_acquisition'],
         'postes': {}},
 'A08_1': {'table_id': 'A08_1',
           'titre': '8/1 Relevé des pertes de valeurs sur créances',
           'dynamic': True,
           'cols': ['designation_debiteur', 'valeur_creance', 'perte_valeur_constituee'],
           'str_cols': ['designation_debiteur'],
           'postes': {}},
 'A08_2': {'table_id': 'A08_2',
           'titre': '8/2 Relevé des pertes de valeurs sur actions et parts sociales',
           'dynamic': True,
           'cols': ['filiale', 'valeur_nominale_debut', 'perte_valeur_constituee', 'valeur_nette_comptable'],
           'str_cols': ['filiale'],
           'postes': {}},
 'A11': {'table_id': 'A11',
         'titre': '11/ Tableau des participations (filiales et entités associées)',
         'dynamic': True,
         'cols': ['filiale_entite_associee',
                  'capitaux_propres',
                  'dont_capital',
                  'quote_part_capital_pct',
                  'resultat_dernier_exercice',
                  'prets_avances',
                  'dividendes_encaisses',
                  'valeur_comptable_titres'],
         'str_cols': ['filiale_entite_associee'],
         'postes': {}},
 'A12': {'table_id': 'A12',
         'titre': '12/ Commissions et courtages, redevances, honoraires, sous-traitance, rémunérations diverses et frais de siège',
         'dynamic': True,
         'cols': ['designation_beneficiaire', 'nif', 'adresse', 'montant_percu'],
         'str_cols': ['designation_beneficiaire', 'nif', 'adresse'],
         'postes': {}},
 'A13': {'table_id': 'A13',
         'titre': '13/ Taxe sur l’activité professionnelle',
         'dynamic': True,
         'cols': ['lieu_paiement_tap', 'ca_imposable', 'ca_exonere', 'tap_acquittee'],
         'str_cols': ['lieu_paiement_tap'],
         'postes': {}}}

# Le tableau A09 contient aussi jusqu'à 4 lignes de déficits antérieurs avec année variable.
for _i in range(1,5):
    TABLE_SCHEMAS["A09"].get("postes",{}).pop(f"deficit_anterieur_{_i}", None)
TABLE_SCHEMAS["A09"]["dynamic_sections"] = {
    "deficits_anterieurs": {"max_rows":4, "cols":["annee","montant"], "str_cols":["annee"]}
}

# Pages physiques du formulaire officiel : une page peut porter plusieurs tableaux logiques.
PAGE_TEMPLATES = {
    "ACTIF":      {"tables":["ACTIF"], "anchors":["bilan actif","actifs non courants","actifs courants","total general actif"]},
    "PASSIF":     {"tables":["PASSIF"],"anchors":["bilan passif","capitaux propres","passifs non courants","passifs courants","total passif"]},
    # TCR_FULL = variante réglementaire/éditeur où les deux parties du compte de résultat sont réunies sur UNE seule page.
    "TCR_FULL":   {"tables":["TCR"],   "anchors":["compte de resultat","i production de l exercice","ii consommations de l exercice","iv excedent brut d exploitation","v resultat operationnel","vi resultat financier","vii resultat ordinaire","ix resultat net de l exercice"]},
    "TCR_P1":     {"tables":["TCR"],   "anchors":["compte de resultat","ventes de marchandises","i production de l exercice","ii consommations de l exercice","iv excedent brut d exploitation"]},
    "TCR_P2":     {"tables":["TCR"],   "anchors":["v resultat operationnel","vi resultat financier","vii resultat ordinaire","ix resultat net de l exercice"]},
    "A01_A02":    {"tables":["A01","A02"],"anchors":["1 tableau des mouvements des stocks","2 tableau de la fluctuation de la production stockee"]},
    "A03_A04":    {"tables":["A03","A04"],"anchors":["3 charges de personnel","4 autres charges et produits operationnels"]},
    "A05_A06":    {"tables":["A05","A06"],"anchors":["5 tableau des amortissements et pertes de valeurs","6 tableau des immobilisations creees ou acquises"]},
    "A07_A08":    {"tables":["A07","A08"],"anchors":["7 tableau des immobilisations cedees","8 tableau des provisions et pertes de valeurs"]},
    "A08_1_A08_2":{"tables":["A08_1","A08_2"],"anchors":["8 1 releve des pertes de valeurs sur creances","8 2 releve des pertes de valeurs sur actions"]},
    "A09":        {"tables":["A09"],"anchors":["9 tableau de determination du resultat fiscal","ii reintegrations","iii deductions","resultat fiscal"]},
    "A10_A11":    {"tables":["A10","A11"],"anchors":["10 tableau d affectation du resultat et des reserves","11 tableau des participations"]},
    "A12_A13":    {"tables":["A12","A13"],"anchors":["12 commissions et courtages","13 taxe sur l activite professionnelle"]},
}

# La page 3 et la page 4 forment un seul TCR logique avec les mêmes quatre colonnes atomiques.
TCR_COLUMNS = ["n_debit", "n_credit", "n1_debit", "n1_credit"]
TCR_P1_ROWS = [
 "ventes_marchandises","produits_fabriques","prestations_services","ventes_travaux","produits_annexes",
 "rabais_remises_ristournes_accordes","chiffre_affaires_net","production_stockee_destockee","production_immobilisee",
 "subvention_exploitation","production_exercice","achats_marchandises_vendues","matieres_premieres","autres_approvisionnements",
 "variation_stocks","achats_etudes_prestations","autres_consommations","rabais_remises_ristournes_obtenus_achats",
 "sous_traitance_generale","locations","entretien_reparations","primes_assurances","personnel_exterieur",
 "remuneration_intermediaires","publicite","deplacements_missions","autres_services",
 "rabais_remises_ristournes_obtenus_services","consommations_exercice","valeur_ajoutee_exploitation",
 "charges_personnel","impots_taxes_assimiles","excedent_brut_exploitation"
]
TCR_P2_ROWS = [
 "autres_produits_operationnels","autres_charges_operationnelles","dotations_amortissements","provisions","pertes_valeur",
 "reprises_pertes_valeur_provisions","resultat_operationnel","produits_financiers","charges_financieres","resultat_financier",
 "resultat_ordinaire","elements_extraordinaires_produits","elements_extraordinaires_charges","resultat_extraordinaire",
 "impots_exigibles_resultats","impots_differes_resultats","resultat_net_exercice"
]
assert set(TCR_P1_ROWS + TCR_P2_ROWS) == set(TABLE_SCHEMAS["TCR"]["postes"])
assert TABLE_SCHEMAS["TCR"]["cols"] == TCR_COLUMNS

# Identifiants uniques du contrat de données. Pour les tables dynamiques, l'index de record sera ajouté à l'exécution.
FIELD_REGISTRY = {}
for table_id, spec in TABLE_SCHEMAS.items():
    if spec.get("dynamic"):
        for col in spec["cols"]:
            fid = f"{table_id}.records[].{col}"
            if fid in FIELD_REGISTRY: raise ValueError("field_id dupliqué: "+fid)
            FIELD_REGISTRY[fid] = {"table_id":table_id,"row_id":"records[]","column_id":col}
    else:
        for row_id in spec.get("postes",{}):
            for col in spec["cols"]:
                fid = f"{table_id}.{row_id}.{col}"
                if fid in FIELD_REGISTRY: raise ValueError("field_id dupliqué: "+fid)
                FIELD_REGISTRY[fid] = {"table_id":table_id,"row_id":row_id,"column_id":col}
    for section_id, section in spec.get("dynamic_sections",{}).items():
        for col in section.get("cols",[]):
            fid=f"{table_id}.{section_id}[].{col}"
            if fid in FIELD_REGISTRY: raise ValueError("field_id dupliqué: "+fid)
            FIELD_REGISTRY[fid]={"table_id":table_id,"row_id":section_id+"[]","column_id":col}

print("✅ Schéma officiel chargé:", len(TABLE_SCHEMAS), "tables |", len(FIELD_REGISTRY), "field_id")
print("✅ TCR verrouillé:", TCR_COLUMNS, "| P1", len(TCR_P1_ROWS), "lignes | P2", len(TCR_P2_ROWS), "lignes | FULL", len(TCR_P1_ROWS)+len(TCR_P2_ROWS), "lignes")


In [ ]:
# CELLULE 6 — ROUTAGE SCHEMA-FIRST (TEXTE D'ABORD, QWEN SEULEMENT SI NECESSAIRE)

def _hits(text_norm, template):
    return sum(a in text_norm for a in PAGE_TEMPLATES[template]["anchors"])

def _tcr_text_shape(text_norm):
    """Signature structurelle du TCR, indépendante des valeurs lues.
    Elle sert uniquement au routage de la page et ne déplace jamais un montant.
    """
    p1_core = [
        "ventes de marchandises", "i production de l exercice",
        "ii consommations de l exercice", "iii valeur ajoutee",
        "iv excedent brut d exploitation"
    ]
    p2_core = [
        "autres produits operationnels", "v resultat operationnel",
        "vi resultat financier", "vii resultat ordinaire",
        "viii resultat extraordinaire", "ix resultat net de l exercice"
    ]
    p1_hits = sum(x in text_norm for x in p1_core)
    p2_hits = sum(x in text_norm for x in p2_core)
    has_title = "compte de resultat" in text_norm
    has_debit_credit = ("debit" in text_norm and "credit" in text_norm)
    has_n1 = ("n 1" in text_norm or "n1" in text_norm)
    return {
        "p1_hits": p1_hits, "p2_hits": p2_hits,
        "has_title": has_title,
        "has_4cols_header": has_debit_credit and has_n1,
        "fiscal_annex": ("reintegrations" in text_norm and "deductions" in text_norm) or "resultat fiscal" in text_norm,
    }

def route_from_text(text):
    t = norm_text(text)
    if not t: return None

    # Annexes d'abord : A09 et A03/A04 doivent gagner contre toute ressemblance locale avec le TCR.
    ordered = ["A01_A02","A03_A04","A05_A06","A07_A08","A08_1_A08_2","A09","A10_A11","A12_A13"]
    for template in ordered:
        anchors = PAGE_TEMPLATES[template]["anchors"]
        hits = sum(a in t for a in anchors)
        if hits >= 1 and (template not in {"A09"} or hits >= 2):
            return template

    if "bilan actif" in t and "total general actif" in t: return "ACTIF"
    if "bilan passif" in t and ("capitaux propres" in t or "total passif" in t): return "PASSIF"

    # TCR : on distingue explicitement les deux mises en page possibles.
    sig = _tcr_text_shape(t)
    if not sig["fiscal_annex"]:
        # Cas 1 : TCR complet sur une seule page. Il faut des signatures des DEUX moitiés.
        if sig["p1_hits"] >= 2 and sig["p2_hits"] >= 2 and (sig["has_title"] or sig["has_4cols_header"]):
            return "TCR_FULL"
        # Cas 2a : première page du TCR sur deux pages.
        if sig["p1_hits"] >= 2 and sig["has_title"] and sig["has_4cols_header"]:
            return "TCR_P1"
        # Cas 2b : deuxième page du TCR; l'entête 4 colonnes peut ne pas être répété.
        if sig["p2_hits"] >= 3:
            return "TCR_P2"
    return None

ROUTE_CHOICES = list(PAGE_TEMPLATES) + ["AUTRE"]
PROMPT_ROUTE = """Tu classes UNE page de la liasse fiscale algérienne Série G n°2.
Réponds UNIQUEMENT en JSON strict: {\"type\":\"...\"}.
Types autorisés: ACTIF, PASSIF, TCR_FULL, TCR_P1, TCR_P2, A01_A02, A03_A04, A05_A06, A07_A08, A08_1_A08_2, A09, A10_A11, A12_A13, AUTRE.
Règle fondamentale: classe la STRUCTURE GLOBALE du formulaire, jamais une ligne isolée.
- ACTIF: BILAN ACTIF avec ACTIFS NON COURANTS + ACTIFS COURANTS + TOTAL GENERAL ACTIF.
- PASSIF: BILAN PASSIF avec CAPITAUX PROPRES + PASSIFS NON-COURANTS + PASSIFS COURANTS + TOTAL PASSIF.
- TCR_FULL: COMPTE DE RESULTAT complet réuni sur UNE page. La page contient des lignes de la partie I-IV ET de la partie V-IX. Les colonnes restent N Débit/N Crédit/N-1 Débit/N-1 Crédit.
- TCR_P1: première partie du COMPTE DE RESULTAT, jusqu'à IV Excédent brut d'exploitation, quatre colonnes N Débit/N Crédit/N-1 Débit/N-1 Crédit.
- TCR_P2: suite du COMPTE DE RESULTAT, lignes V Résultat opérationnel, VI Résultat financier, VII Résultat ordinaire, VIII Résultat extraordinaire, IX Résultat net. La page peut ne pas répéter l'entête des quatre colonnes.
- A09: tableau 9 Détermination du résultat fiscal avec Réintégrations + Déductions; il reste A09 même s'il contient « Résultat net de l'exercice (Compte de résultat) ».
- A03_A04 peut contenir « Autres produits opérationnels » ou « Autres charges opérationnelles » mais ce n'est PAS le TCR.
En cas de doute: AUTRE."""

# Validation structurelle uniforme : elle s'applique à TOUS les gabarits, pas à un cas PASSIF particulier.
def structure_signature(page_template, rowcodes):
    rows=set(rowcodes or [])
    if page_template=="ACTIF":
        return ("total_general_actif" in rows and
                bool(rows & {"immobilisations_incorporelles","terrains","immobilisations_financieres","total_actif_non_courant"}) and
                bool(rows & {"stocks_encours","clients","tresorerie_actif","total_actif_courant"}))
    if page_template=="PASSIF":
        return ("total_general_passif" in rows and
                bool(rows & {"capital_emis","primes_reserves","resultat_net_passif","total_capitaux_propres"}) and
                bool(rows & {"emprunts_dettes_financieres","fournisseurs_rattaches","autres_dettes","total_passifs_courants"}))
    p1_sig={"chiffre_affaires_net","production_exercice","consommations_exercice","valeur_ajoutee_exploitation","excedent_brut_exploitation"}
    p2_sig={"resultat_operationnel","resultat_financier","resultat_ordinaire","resultat_extraordinaire","resultat_net_exercice"}
    if page_template=="TCR_P1": return len(rows & p1_sig) >= 2
    if page_template=="TCR_P2": return len(rows & p2_sig) >= 2
    if page_template=="TCR_FULL":
        # Une page FULL doit confirmer les deux moitiés; jamais uniquement quelques lignes de l'une des deux.
        return len(rows & p1_sig) >= 1 and len(rows & p2_sig) >= 1
    return True

print("✅ Routeur schema-first prêt — TCR 1 page (FULL) ou 2 pages (P1+P2)")


In [ ]:
# CELLULE 7 — RENDU PDF + ENTETES + INFERENCE vLLM
def resize_image(img, max_side):
    w,h=img.size
    if max(w,h)<=max_side: return img
    r=max_side/max(w,h)
    return img.resize((max(1,int(w*r)), max(1,int(h*r))), Image.LANCZOS)

def estimate_skew(img):
    small=img.convert("L").copy(); small.thumbnail((420,420))
    def score(a):
        r=np.array(small.rotate(a,expand=True,fillcolor=255))<128
        proj=r.sum(axis=1); return float((proj**2).sum())
    coarse=max(range(-8,9,2),key=score)
    best=max([coarse-1,coarse-.5,coarse,coarse+.5,coarse+1],key=score)
    return best if abs(best)>=1 else 0.0

def deskew(img):
    a=estimate_skew(img)
    return (img.rotate(a,expand=True,fillcolor=(255,255,255),resample=Image.BICUBIC),a) if a else (img,0.0)

def is_blank(img):
    small=img.convert("L").copy(); small.thumbnail((500,500)); arr=np.asarray(small)
    return (arr>245).sum()/arr.size >= BLANK_THRESHOLD

def pdf_manifest(path):
    doc=fitz.open(path); out=[]
    for i,p in enumerate(doc):
        r=p.rect
        out.append({"index":i,"text":p.get_text("text") or "","largeur_pdf":float(r.width),"hauteur_pdf":float(r.height)})
    doc.close(); return out

def render_page(path, idx, zoom, max_side, do_deskew=False):
    doc=fitz.open(path); pix=doc.load_page(idx).get_pixmap(matrix=fitz.Matrix(zoom,zoom),alpha=False); doc.close()
    img=Image.frombytes("RGB",[pix.width,pix.height],pix.samples); angle=0.0
    if do_deskew: img,angle=deskew(img)
    img=resize_image(img,max_side)
    return {"index":idx,"image":img,"largeur_px":img.width,"hauteur_px":img.height,"rotation_estimee":angle}

def render_pages(path, indices, zoom, max_side, do_deskew=False):
    indices=list(indices)
    if not indices: return []
    def one(i): return render_page(path,i,zoom,max_side,do_deskew)
    with ThreadPoolExecutor(max_workers=min(PDF_WORKERS,len(indices))) as ex:
        out=list(ex.map(one,indices))
    return sorted(out,key=lambda x:x["index"])

def render_header_page(path, idx, zoom=IDENTITY_HEADER_ZOOM, max_side=IDENTITY_HEADER_MAX_SIDE, fraction=IDENTITY_HEADER_FRACTION):
    doc=fitz.open(path); page=doc.load_page(idx); r=page.rect
    clip=fitz.Rect(r.x0,r.y0,r.x1,r.y0+r.height*float(fraction))
    pix=page.get_pixmap(matrix=fitz.Matrix(zoom,zoom),clip=clip,alpha=False); doc.close()
    img=Image.frombytes("RGB",[pix.width,pix.height],pix.samples)
    img=resize_image(img,max_side)
    return {"index":idx,"image":img,"largeur_px":img.width,"hauteur_px":img.height}

def render_header_pages(path, indices):
    indices=list(indices)
    if not indices: return []
    with ThreadPoolExecutor(max_workers=min(PDF_WORKERS,len(indices))) as ex:
        out=list(ex.map(lambda i: render_header_page(path,i),indices))
    return sorted(out,key=lambda x:x["index"])

def identity_from_text_layer(text):
    raw=str(text or "")
    out={"identity_source":"TEXTE_PDF"}
    # NIF : recherche près du libellé puis fallback sur un bloc isolé de 15 chiffres.
    m=re.search(r"N\s*[. ]*I\s*[. ]*F\s*[:\-]?\s*([0-9][0-9 .\-]{13,28})",raw,re.I)
    if m:
        n=norm_nif(m.group(1)); out["nif"]=m.group(1).strip(); out["nif_15"]=n
    if not out.get("nif_15"):
        for cand in re.findall(r"(?<!\d)(?:\d[ .\-]?){15}(?!\d)",raw):
            n=norm_nif(cand)
            if n: out["nif"]=cand.strip(); out["nif_15"]=n; break
    lines=[x.strip() for x in raw.splitlines() if x.strip()]
    for i,line in enumerate(lines):
        if re.search(r"D[ée]signation\s+de\s+l[’']?entreprise",line,re.I):
            tail=re.split(r"[:：]",line,maxsplit=1)
            val=tail[1].strip(" .…_-") if len(tail)>1 else ""
            if not val and i+1<len(lines):
                nxt=lines[i+1].strip(" .…_-")
                if nxt and not re.search(r"^(Activit[ée]|Adresse|Exercice|N\.?I\.?F)",nxt,re.I): val=nxt
            if val: out["entreprise"]=val
            break
    # Exercice : utilisé comme appoint, jamais pour forcer une identité.
    ym=re.findall(r"(?<!\d)(20\d{2}|19\d{2})(?!\d)",raw)
    if ym: out["exercice"]=ym[-1]
    return normalize_page_identity(out)

def parse_json_strictish(text):
    if not text: return {}
    s=text.strip().replace("```json","").replace("```","").strip()
    try: return json.loads(s)
    except Exception: pass
    a,b=s.find("{"),s.rfind("}")
    if a>=0 and b>a:
        try: return json.loads(s[a:b+1])
        except Exception: pass
    return {}

def apply_template(messages):
    try: return processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    except TypeError: return processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)

def build_inputs(prompt,images):
    out=[]
    for img in images:
        msgs=[{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":prompt}]}]
        out.append({"prompt":apply_template(msgs),"multi_modal_data":{"image":img}})
    return out

def ask_batch(prompt,images,sampling):
    if not images: return []
    t0=time.time(); outputs=llm.generate(build_inputs(prompt,images),sampling,use_tqdm=False); elapsed=time.time()-t0
    res=[]
    for o in outputs:
        res.append({"text":o.outputs[0].text,"tokens_in":len(o.prompt_token_ids),"tokens_out":len(o.outputs[0].token_ids),"elapsed":elapsed/max(1,len(images))})
    return res

def ask_single(prompt,img,sampling): return ask_batch(prompt,[img],sampling)[0]


In [ ]:
# CELLULE 8 — CHARGEMENT QWEN / vLLM PERFORMANCE V26
# IMPORTANT : les variables d'environnement sont fixées AVANT import vLLM.
os.environ["VLLM_USE_DEEP_GEMM"]="0"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"]="0"
os.environ["VLLM_ATTENTION_BACKEND"]="FLASH_ATTN"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"]="spawn"

if not torch.cuda.is_available(): raise RuntimeError("CUDA indisponible")
name=torch.cuda.get_device_name(0); props=torch.cuda.get_device_properties(0); total_gib=props.total_memory/1024**3
free_b,_=torch.cuda.mem_get_info(); print(f"GPU={name} | VRAM={total_gib:.1f} GiB | libre={free_b/1024**3:.1f} GiB")

if total_gib < 50:
    VLLM_GPU_UTIL=.96; VLLM_MAX_MODEL_LEN=5120; VLLM_MAX_NUM_SEQS=4; VLLM_MAX_BATCHED_TOKENS=12288
else:
    VLLM_GPU_UTIL=.92; VLLM_MAX_MODEL_LEN=5120; VLLM_MAX_NUM_SEQS=8; VLLM_MAX_BATCHED_TOKENS=32768

from vllm import LLM, SamplingParams
from transformers import AutoProcessor

t0=time.time()
processor=AutoProcessor.from_pretrained(MODEL_PATH,trust_remote_code=True,min_pixels=4*32*32,max_pixels=2000*32*32)
processor.tokenizer.padding_side="left"
llm=LLM(model=MODEL_PATH,dtype="bfloat16",quantization="fp8",gpu_memory_utilization=VLLM_GPU_UTIL,
        max_model_len=VLLM_MAX_MODEL_LEN,max_num_seqs=VLLM_MAX_NUM_SEQS,max_num_batched_tokens=VLLM_MAX_BATCHED_TOKENS,
        kv_cache_dtype="fp8",trust_remote_code=True,enforce_eager=False,enable_prefix_caching=True,
        limit_mm_per_prompt={"image":1},mm_processor_kwargs={"min_pixels":4*32*32,"max_pixels":2000*32*32})
SAMPLING_ROUTE=SamplingParams(temperature=0.0,max_tokens=48,repetition_penalty=1.0)
SAMPLING_IDENTITY=SamplingParams(temperature=0.0,max_tokens=160,repetition_penalty=1.0)
SAMPLING_EXTRACT=SamplingParams(temperature=0.0,max_tokens=2200,repetition_penalty=1.0)  # marge pour TCR_FULL; arrêt naturel dès fin du JSON
print("✅ Qwen chargé en",round(time.time()-t0,1),"s | DeepGEMM OFF | eager=False")


In [ ]:
# CELLULE 9 — PROMPTS FERMES PAR SCHEMA + IDENTITE + DECODAGE
# Qwen ne reçoit jamais les field_id finaux : seulement des ids courts appartenant au gabarit courant.
def rows_for_template(page_template):
    if page_template=="ACTIF": return list(TABLE_SCHEMAS["ACTIF"]["postes"])
    if page_template=="PASSIF": return list(TABLE_SCHEMAS["PASSIF"]["postes"])
    if page_template=="TCR_P1": return TCR_P1_ROWS
    if page_template=="TCR_P2": return TCR_P2_ROWS
    if page_template=="TCR_FULL": return TCR_P1_ROWS + TCR_P2_ROWS
    return []

def table_for_template(page_template):
    return "TCR" if page_template.startswith("TCR_") else page_template

def compact_contract(page_template):
    table=table_for_template(page_template); spec=TABLE_SCHEMAS[table]; rows=rows_for_template(page_template)
    short_to_row={f"r{i:02d}":rc for i,rc in enumerate(rows,1)}
    row_to_short={v:k for k,v in short_to_row.items()}
    return table,spec,short_to_row,row_to_short

def build_extract_prompt(page_template):
    table,spec,short_to_row,_=compact_contract(page_template)
    cols=spec["cols"]
    lines=[
        f"Tu lis UNIQUEMENT le gabarit officiel {page_template} de la liasse fiscale algérienne Série G n°2.",
        "TRANSCRIPTION VISUELLE UNIQUEMENT: ne calcule rien, ne complète rien, ne déplace aucune valeur.",
        "Une case vide ou illisible = null. N'invente jamais une valeur.",
        "Réponds UNIQUEMENT en JSON strict:",
        '{"entete":{"entreprise":null,"nif":null,"exercice":null,"exercice_du":null,"exercice_au":null},"v":{"r01":[...]}}',
        "Retourne uniquement les ids rXX autorisés ayant au moins une valeur non nulle.",
        "Ordre EXACT des colonnes: ["+", ".join(cols)+"]",
    ]
    if table=="TCR":
        lines += [
            "REGLE CRITIQUE TCR: la liste a TOUJOURS 4 positions dans cet ordre immuable:",
            "1=N DEBIT, 2=N CREDIT, 3=N-1 DEBIT, 4=N-1 CREDIT.",
            "TCR_P1, TCR_P2 et TCR_FULL utilisent exactement ces 4 colonnes; TCR_P2 peut ne pas répéter l entête.",
            "Si page_template=TCR_FULL, lis TOUTES les lignes visibles des parties I-IV puis V-IX dans UNE SEULE réponse.",
            "Ne transforme jamais Débit en Crédit; ne transforme jamais N en N-1; si la case est incertaine mets null.",
        ]
    lines.append("IDs autorisés:")
    for sid,rc in short_to_row.items(): lines.append(f"{sid} = {spec['postes'][rc]}")
    return "\n".join(lines)

PROMPTS_EXTRACT={t:build_extract_prompt(t) for t in ("ACTIF","PASSIF","TCR_P1","TCR_P2","TCR_FULL")}

def decode_compact(page_template, data):
    table,spec,short_to_row,_=compact_contract(page_template); cols=spec["cols"]
    raw=(data or {}).get("v") or {}; out={}; unknown=[]
    if not isinstance(raw,dict): raw={}
    for sid,vals in raw.items():
        if sid not in short_to_row:
            unknown.append(sid); continue
        if isinstance(vals,dict): arr=[vals.get(c) for c in cols]
        elif isinstance(vals,(list,tuple)): arr=list(vals)
        else: continue
        arr=(arr+[None]*len(cols))[:len(cols)]
        normalized={c:norm_amount(arr[i]) for i,c in enumerate(cols)}
        if any(v is not None for v in normalized.values()): out[short_to_row[sid]]=normalized
    ent=(data or {}).get("entete") if isinstance((data or {}).get("entete"),dict) else {}
    return table,out,ent,unknown

def count_values(rows): return sum(v is not None for vals in rows.values() for v in vals.values())



PROMPT_IDENTITY = """Lis UNIQUEMENT l'entête de cette page de liasse fiscale algérienne.
Retourne UNIQUEMENT un JSON strict de cette forme :
{"entreprise":null,"nif":null,"exercice":null,"exercice_du":null,"exercice_au":null}
- entreprise = raison sociale / Désignation de l'entreprise telle qu'imprimée ;
- nif = N.I.F tel qu'imprimé, sans en inventer un ;
- une information absente ou illisible = null ;
- ne lis pas les montants du tableau et ne déduis rien.
"""

print("✅ Prompts schema-first + identité page construits")


In [ ]:
# CELLULE 10 — JSON CANONIQUE / IDENTITE PAGE PAR PAGE
def page_tables_from_template(template): return PAGE_TEMPLATES.get(template,{}).get("tables",[])

def observations_for_page(table,rows,document,page_no):
    obs=[]
    for row_id,vals in rows.items():
        for col,val in vals.items():
            if val is None: continue
            obs.append({
                "field_id":f"{table}.{row_id}.{col}","table_id":table,"row_id":row_id,"column_id":col,"value":val,"unit":"DZD",
                "source":{"document":document,"page":page_no}
            })
    return obs

def classification_page(manifest_page,template,document,route_source):
    n=manifest_page["index"]+1
    blank=template=="BLANCHE"
    return {
        "page_id":f"p{n:03d}","pdf_page_index":manifest_page["index"],"numero_page_scannee":n,"fichier_source":document,
        "classification":{"page_template":template,"types":page_tables_from_template(template),"route_source":route_source,
                          "page_utile":not blank,"page_blanche":blank,"extraction_active":False},
        "entete_page":{},"donnees":{},"observations":[],"statut_extraction":{"statut":"BLANCHE" if blank else "CLASSEE_NON_EXTRAITE"},
        "tokens_in":0,"tokens_out":0,"temps_s":0.0
    }

def extracted_page(rendered,page_template,table,rows,entete,document,route_source,rep,structure_ok,unknown_ids):
    n=rendered["index"]+1
    return {
        "page_id":f"p{n:03d}","pdf_page_index":rendered["index"],"numero_page_scannee":n,"fichier_source":document,
        "classification":{"page_template":page_template,"types":[table],"route_source":route_source,
                          "page_utile":True,"page_blanche":False,"extraction_active":True,"structure_ok":bool(structure_ok)},
        "image":{"largeur_px":rendered.get("largeur_px"),"hauteur_px":rendered.get("hauteur_px"),"rotation_estimee":rendered.get("rotation_estimee",0.0)},
        "entete_page":normalize_page_identity(entete),"donnees":{table:rows},"observations":observations_for_page(table,rows,document,n),
        "statut_extraction":{"statut":"OK" if structure_ok else "A_REVOIR_STRUCTURE","nb_champs_extraits":count_values(rows),"unknown_ids_ignores":unknown_ids},
        "tokens_in":rep.get("tokens_in",0),"tokens_out":rep.get("tokens_out",0),"temps_s":round(rep.get("elapsed",0.0),2)
    }

def page_identity_record(p):
    e=normalize_page_identity(p.get("entete_page") or {})
    return {"page":p.get("numero_page_scannee"),"template":(p.get("classification") or {}).get("page_template"),
            "entreprise":e.get("entreprise"),"nif_15":e.get("nif_15"),"identity_source":e.get("identity_source")}

def _name_medoid(values):
    vals=[v for v in values if v]
    if not vals: return None
    if len(vals)==1: return vals[0]
    return max(vals,key=lambda x: sum((company_similarity(x,y) or 0.0) for y in vals))

def build_identity(pages, filename):
    ents=[normalize_page_identity(p.get("entete_page") or {}) for p in pages if not (p.get("classification") or {}).get("page_blanche")]
    names=[e.get("entreprise") for e in ents if e.get("entreprise")]
    nifs=[e.get("nif_15") for e in ents if e.get("nif_15")]
    entreprise=_name_medoid(names)
    nif=Counter(nifs).most_common(1)[0][0] if nifs else None
    def majority(field):
        vals=[str(e.get(field)).strip() for e in ents if e.get(field) not in (None,"")]
        return Counter(vals).most_common(1)[0][0] if vals else None
    du=majority("exercice_du"); au=majority("exercice_au"); ex=majority("exercice")
    year=parse_year(au) or parse_year(ex)
    if not year:
        ys=re.findall(r"(?<!\d)(20\d{2}|19\d{2})(?!\d)",filename); year=int(ys[-1]) if ys else None
    return {"entreprise":entreprise,"nif_15":nif,"exercice_du":du,"exercice_au":au,"exercice":ex,"annee":year,
            "pages_identitees":sum(bool(e.get("entreprise") or e.get("nif_15")) for e in ents),"pages_total":len(ents)}

def enrich_page_identities(pdf_path,pages,manifest):
    """Complète raison sociale/NIF pour chaque page utile. Priorité : entête déjà lu, couche texte, puis crop Qwen.
    Le crop est très petit et traité en batch pour limiter le coût.
    """
    by_idx={p["pdf_page_index"]:p for p in pages}
    need=[]
    for mp in manifest:
        idx=mp["index"]; p=by_idx[idx]
        if (p.get("classification") or {}).get("page_blanche"): continue
        existing=normalize_page_identity(p.get("entete_page") or {})
        txt=identity_from_text_layer(mp.get("text",""))
        merged=merge_identity(existing,txt)
        p["entete_page"]=merged
        if not merged.get("entreprise") or not merged.get("nif_15"):
            need.append(idx)
    t_render=t_qwen=0.0; ti=to=0
    if need:
        t0=time.time(); headers=render_header_pages(pdf_path,need); t_render=time.time()-t0
        for batch in chunks(headers,IDENTITY_BATCH_SIZE):
            t0=time.time(); reps=ask_batch(PROMPT_IDENTITY,[x["image"] for x in batch],SAMPLING_IDENTITY); t_qwen+=time.time()-t0
            for h,rep in zip(batch,reps):
                raw=parse_json_strictish(rep.get("text")); q=normalize_page_identity(raw); q["identity_source"]="HEADER_QWEN"
                p=by_idx[h["index"]]
                # Le résultat spécialisé d'entête complète les champs manquants; un NIF 15 chiffres spécialisé est prioritaire.
                old=normalize_page_identity(p.get("entete_page") or {})
                merged=merge_identity(q,old)
                if old.get("entreprise") and q.get("entreprise"):
                    # conserver les deux lectures pour audit en cas de divergence OCR/vision
                    merged["entreprise_extraction_table"] = old.get("entreprise")
                p["entete_page"]=merged
                ti+=rep.get("tokens_in",0); to+=rep.get("tokens_out",0)
    return {"identity_render":round(t_render,2),"identity_qwen":round(t_qwen,2),"identity_pages_qwen":len(need)},ti,to

def build_document(pdf_path,pages,timings,tokens):
    ident=build_identity(pages,pdf_path.name)
    return {
        "schema_version":SCHEMA_VERSION,"schema_registry_version":SCHEMA_REGISTRY_VERSION,
        "document":{"fichier_source":pdf_path.name,"chemin_source":str(pdf_path),"modele_extraction":MODEL_NAME,"mode_execution":"V26_SCHEMA_FIRST_IDENTITY_ALERTS","timings_s":timings},
        "identite":ident,"identites_pages":[page_identity_record(p) for p in pages],"pages":pages,"observations":[o for p in pages for o in p.get("observations",[])],
        "tokens":{"input":tokens[0],"output":tokens[1]},"controles":{}
    }


In [ ]:
# CELLULE 11 — PIPELINE EXTRACTION + IDENTITE PAGE PAR PAGE + CACHE V25
def client_id_from_pdf(pdf_path):
    rel=pdf_path.relative_to(INPUT_DIR); return rel.parts[0] if len(rel.parts)>1 else "_RACINE_"

def json_path_for_pdf(pdf_path):
    d=JSON_DIR/client_id_from_pdf(pdf_path); d.mkdir(parents=True,exist_ok=True); return d/(pdf_path.stem+".json")

def legacy_json_for_pdf(pdf_path):
    cid=client_id_from_pdf(pdf_path)
    for root in LEGACY_JSON_DIRS:
        p=root/cid/(pdf_path.stem+".json")
        if p.exists(): return p
    return None

def prime_json_cache_from_legacy(pdfs):
    copied=0
    if FORCE_REEXTRACT: return copied
    for p in pdfs:
        dst=json_path_for_pdf(p)
        if dst.exists(): continue
        old=legacy_json_for_pdf(p)
        if old and old.exists():
            shutil.copy2(old,dst); copied+=1
    return copied

def route_with_qwen(pdf_path,manifest,routes,unresolved):
    t_render=t_route=0.0; ti=to=0
    if not unresolved: return t_render,t_route,ti,to
    t0=time.time(); minis=render_pages(pdf_path,unresolved,PDF_ROUTER_ZOOM,ROUTER_MAX_SIDE,False); t_render+=time.time()-t0
    nonblank=[]
    for q in minis:
        if is_blank(q["image"]): routes[q["index"]] = ("BLANCHE","MINI_BLANK")
        else: nonblank.append(q)
    if nonblank:
        t0=time.time(); reps=[]
        for b in chunks(nonblank,CLASSIF_BATCH_SIZE): reps += ask_batch(PROMPT_ROUTE,[x["image"] for x in b],SAMPLING_ROUTE)
        t_route+=time.time()-t0
        for q,rep in zip(nonblank,reps):
            d=parse_json_strictish(rep["text"]); typ=str(d.get("type") or "AUTRE").upper().strip()
            if typ not in ROUTE_CHOICES: typ="AUTRE"
            routes[q["index"]]=(typ,"MINI_QWEN"); ti+=rep["tokens_in"]; to+=rep["tokens_out"]
    return t_render,t_route,ti,to

pdfs=sorted(INPUT_DIR.rglob("*.pdf")) if INPUT_DIR.exists() else []
_reused=prime_json_cache_from_legacy(pdfs)
log(f"V26 cache — {_reused} JSON V25 réutilisé(s)")
a_traiter=[p for p in pdfs if FORCE_REEXTRACT or not json_path_for_pdf(p).exists()]
log(f"V26 extraction — à traiter={len(a_traiter)} | déjà disponibles={len(pdfs)-len(a_traiter)}")
index_records=[]

for num,pdf_path in enumerate(a_traiter,1):
    prefix=f"[{num:04d}/{len(a_traiter):04d}] "; cid=client_id_from_pdf(pdf_path); tpdf=time.time()
    try:
        tok_in=tok_out=0; t_render=t_route=t_extract=0.0; retries=0
        t0=time.time(); manifest=pdf_manifest(pdf_path); t_manifest=time.time()-t0
        routes={}; unresolved=[]
        for p in manifest:
            typ=route_from_text(p.get("text",""))
            if typ: routes[p["index"]]=(typ,"TEXTE_PDF")
            else: unresolved.append(p["index"])
        dr,dq,di,do=route_with_qwen(pdf_path,manifest,routes,unresolved); t_render+=dr; t_route+=dq; tok_in+=di; tok_out+=do

        target=[i for i,(typ,src) in routes.items() if table_for_template(typ) in ACTIVE_TABLES and typ in PROMPTS_EXTRACT]
        counts=Counter(routes.get(i,("AUTRE","DEFAULT"))[0] for i in range(len(manifest)))
        log(prefix+pdf_path.name+" | route "+str(dict(counts)))

        t0=time.time(); rendered=render_pages(pdf_path,target,PDF_EXTRACT_ZOOM,EXTRACT_MAX_SIDE,True); t_render+=time.time()-t0
        by_idx={p["index"]:p for p in rendered}; extracted={}
        for template in ("ACTIF","PASSIF","TCR_FULL","TCR_P1","TCR_P2"):
            plist=[by_idx[i] for i in target if i in by_idx and routes[i][0]==template]
            if not plist: continue
            prompt=PROMPTS_EXTRACT[template]
            for batch in chunks(plist,GPU_BATCH_SIZE):
                t0=time.time(); reps=ask_batch(prompt,[p["image"] for p in batch],SAMPLING_EXTRACT); t_extract+=time.time()-t0
                for p,rep in zip(batch,reps):
                    tok_in+=rep["tokens_in"]; tok_out+=rep["tokens_out"]
                    raw=parse_json_strictish(rep["text"]); table,rows,ent,unknown=decode_compact(template,raw); nvals=count_values(rows)
                    if nvals < MIN_VALUES_RETRY[template]:
                        t1=time.time(); hd=render_page(pdf_path,p["index"],3.0,RETRY_MAX_SIDE,True); t_render+=time.time()-t1
                        rep2=ask_single(prompt,hd["image"],SAMPLING_EXTRACT); t_extract+=rep2["elapsed"]; tok_in+=rep2["tokens_in"]; tok_out+=rep2["tokens_out"]; retries+=1
                        raw2=parse_json_strictish(rep2["text"]); table2,rows2,ent2,unknown2=decode_compact(template,raw2)
                        if count_values(rows2)>nvals: p,rep,table,rows,ent,unknown=hd,rep2,table2,rows2,ent2,unknown2
                    sig=structure_signature(template,rows.keys())
                    # Fail closed uniquement si le routage venait de Qwen ET la structure extraite ne confirme pas le gabarit.
                    # Si le titre officiel était présent dans la couche texte, on conserve l'extraction et on marque la signature.
                    if routes[p["index"]][1]=="MINI_QWEN" and not sig:
                        log(prefix+f"page {p['index']+1}: {template} non confirmé -> AUTRE/A_REVOIR")
                        routes[p["index"]] = ("AUTRE","REJET_STRUCTURE_SCHEMA")
                        continue
                    extracted[p["index"]]=extracted_page(p,template,table,rows,ent,pdf_path.name,routes[p["index"]][1],rep,sig,unknown)

        pages=[]
        for mp in manifest:
            idx=mp["index"]
            if idx in extracted: pages.append(extracted[idx])
            else:
                typ,src=routes.get(idx,("AUTRE","DEFAULT")); pages.append(classification_page(mp,typ,pdf_path.name,src))
        pages.sort(key=lambda x:x["pdf_page_index"])
        ident_t,ii,io=enrich_page_identities(pdf_path,pages,manifest); tok_in+=ii; tok_out+=io
        total=time.time()-tpdf
        timings={"manifest_texte":round(t_manifest,2),"rendu_images":round(t_render,2),"route_qwen":round(t_route,2),"extraction_qwen":round(t_extract,2),
                 **ident_t,"retry_hd":retries,"total":round(total,2)}
        doc=build_document(pdf_path,pages,timings,(tok_in,tok_out)); doc["document"]["client_dossier"]=cid
        out=json_path_for_pdf(pdf_path); out.write_text(json.dumps(doc,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
        index_records.append({"client":cid,"fichier":pdf_path.name,"json":str(out),"timings_s":timings,"tokens":tok_in+tok_out})
        log(prefix+f"OK {pdf_path.name} total={total:.1f}s extract={t_extract:.1f}s route={t_route:.1f}s")
        del rendered,by_idx; gc.collect()
    except Exception as e:
        log(prefix+"ERREUR "+pdf_path.name+" — "+type(e).__name__+": "+str(e))
        with open(LOG_PATH,"a",encoding="utf-8") as f: f.write(traceback.format_exc()+"\n")

with open(INDEX_PATH,"w",encoding="utf-8") as f:
    for r in index_records: f.write(json.dumps(r,ensure_ascii=False,default=str)+"\n")
print("✅ Extraction V26 terminée — identité page par page incluse")


In [ ]:
# CELLULE 12 — MOTEUR DE CONTROLES DETERMINISTES (DZD)
# Les valeurs observées ne sont jamais remplacées par les valeurs calculées.
FORMULAS = [
 ("ACTIF","immobilisations_corporelles",["montant_brut","amortissements_provisions_pertes","net_n"],["terrains","batiments","autres_immobilisations_corporelles"]),
 ("ACTIF","immobilisations_financieres",["montant_brut","amortissements_provisions_pertes","net_n"],["titres_mis_en_equivalence","autres_participations_creances","autres_titres_immobilises","prets_actifs_financiers_non_courants","impots_differes_actif"]),
 ("ACTIF","creances_emplois_assimiles",["montant_brut","amortissements_provisions_pertes","net_n"],["clients","autres_debiteurs","impots_assimiles_actif","autres_creances_assimiles"]),
 ("ACTIF","disponibilites_assimiles",["montant_brut","amortissements_provisions_pertes","net_n"],["placements_financiers_courants","tresorerie_actif"]),
 ("ACTIF","total_actif_non_courant",["montant_brut","amortissements_provisions_pertes","net_n"],["ecarts_acquisition_goodwill","immobilisations_incorporelles","terrains","batiments","autres_immobilisations_corporelles","immobilisations_en_concession","immobilisations_en_cours","titres_mis_en_equivalence","autres_participations_creances","autres_titres_immobilises","prets_actifs_financiers_non_courants","impots_differes_actif"]),
 ("ACTIF","total_actif_courant",["montant_brut","amortissements_provisions_pertes","net_n"],["stocks_encours","clients","autres_debiteurs","impots_assimiles_actif","autres_creances_assimiles","placements_financiers_courants","tresorerie_actif"]),
 ("ACTIF","total_general_actif",["montant_brut","amortissements_provisions_pertes","net_n"],["total_actif_non_courant","total_actif_courant"]),
 ("PASSIF","total_capitaux_propres",["n","n1"],["capital_emis","primes_reserves","ecart_reevaluation","ecart_equivalence","resultat_net_passif","report_a_nouveau","part_societe_consolidante","part_minoritaires"],{"capital_non_appele":-1}),
 ("PASSIF","total_passifs_non_courants",["n","n1"],["emprunts_dettes_financieres","impots_differes_provisionnes","autres_dettes_non_courantes","provisions_produits_avance"]),
 ("PASSIF","total_passifs_courants",["n","n1"],["fournisseurs_rattaches","impots_passif","autres_dettes","tresorerie_passif"]),
 ("PASSIF","total_general_passif",["n","n1"],["total_capitaux_propres","total_passifs_non_courants","total_passifs_courants"]),
]
TCR_FORMULAS = {
 "chiffre_affaires_net":["ventes_marchandises","produits_fabriques","prestations_services","ventes_travaux","produits_annexes","rabais_remises_ristournes_accordes"],
 "production_exercice":["chiffre_affaires_net","production_stockee_destockee","production_immobilisee","subvention_exploitation"],
 "consommations_exercice":["achats_marchandises_vendues","matieres_premieres","autres_approvisionnements","variation_stocks","achats_etudes_prestations","autres_consommations","rabais_remises_ristournes_obtenus_achats","sous_traitance_generale","locations","entretien_reparations","primes_assurances","personnel_exterieur","remuneration_intermediaires","publicite","deplacements_missions","autres_services","rabais_remises_ristournes_obtenus_services"],
 "valeur_ajoutee_exploitation":["production_exercice","consommations_exercice"],
 "excedent_brut_exploitation":["valeur_ajoutee_exploitation","charges_personnel","impots_taxes_assimiles"],
 "resultat_operationnel":["excedent_brut_exploitation","autres_produits_operationnels","autres_charges_operationnelles","dotations_amortissements","provisions","pertes_valeur","reprises_pertes_valeur_provisions"],
 "resultat_financier":["produits_financiers","charges_financieres"],
 "resultat_ordinaire":["resultat_operationnel","resultat_financier"],
 "resultat_extraordinaire":["elements_extraordinaires_produits","elements_extraordinaires_charges"],
 "resultat_net_exercice":["resultat_ordinaire","resultat_extraordinaire","impots_exigibles_resultats","impots_differes_resultats"],
}

def index_doc(doc):
    idx={}
    for p in doc.get("pages",[]):
        for table,rows in (p.get("donnees") or {}).items():
            if not isinstance(rows,dict): continue
            for rc,vals in rows.items():
                if isinstance(vals,dict): idx.setdefault((table,rc),{}).update({k:v for k,v in vals.items() if v is not None})
    return idx

def tcr_net(vals, period="n"):
    d=vals.get(period+"_debit"); c=vals.get(period+"_credit")
    if d is None and c is None: return None
    return float(c or 0)-float(d or 0)

def add_control(out,kind,table,row,col,observed,calc,source=None):
    if observed is None or calc is None: return
    gap=float(observed)-float(calc); out.append({"kind":kind,"table_id":table,"row_id":row,"column_id":col,
        "observed_dzd":observed,"calculated_dzd":calc,"difference_dzd":round(gap,2),"level":ecart_level(gap),"source":source})

def run_document_controls(doc):
    idx=index_doc(doc); out=[]
    for formula in FORMULAS:
        table,target,cols,components=formula[:4]; overrides=formula[4] if len(formula)>4 else {}
        for col in cols:
            observed=(idx.get((table,target)) or {}).get(col)
            if observed is None: continue
            vals=[]; missing=False
            for rc in components:
                v=(idx.get((table,rc)) or {}).get(col)
                if v is None: missing=True; break
                vals.append(float(v)*float(overrides.get(rc,1)))
            # capital non appelé est un composant soustractif même s'il n'est pas dans la liste positive.
            for rc,mul in overrides.items():
                if rc in components: continue
                v=(idx.get((table,rc)) or {}).get(col)
                if v is None: missing=True; break
                vals.append(float(v)*float(mul))
            if not missing: add_control(out,"FORMULA_SCHEMA",table,target,col,observed,sum(vals))
    # ACTIF: brut - amort = net N ligne par ligne.
    for (table,rc),vals in idx.items():
        if table=="ACTIF" and vals.get("net_n") is not None and vals.get("montant_brut") is not None and vals.get("amortissements_provisions_pertes") is not None:
            add_control(out,"FORMULA_LINE","ACTIF",rc,"net_n",vals["net_n"],float(vals["montant_brut"])-float(vals["amortissements_provisions_pertes"]))
    # TCR: contrôle des soldes calculés sur N et N-1; les 4 cellules brutes restent intactes.
    for target,components in TCR_FORMULAS.items():
        for period in ("n","n1"):
            obs=tcr_net(idx.get(("TCR",target),{}),period)
            if obs is None: continue
            parts=[]; missing=False
            for rc in components:
                v=tcr_net(idx.get(("TCR",rc),{}),period)
                if v is None: missing=True; break
                parts.append(v)
            if not missing: add_control(out,"TCR_NET_SCHEMA","TCR",target,period+"_net",obs,sum(parts))
    # Contrôles croisés majeurs.
    a=(idx.get(("ACTIF","total_general_actif")) or {}).get("net_n"); p=(idx.get(("PASSIF","total_general_passif")) or {}).get("n")
    if a is not None and p is not None: add_control(out,"CROSS","BILAN","actif_passif","N",a,p)
    tn=tcr_net(idx.get(("TCR","resultat_net_exercice"),{}),"n"); pn=(idx.get(("PASSIF","resultat_net_passif")) or {}).get("n")
    if tn is not None and pn is not None: add_control(out,"CROSS","RESULTAT","tcr_passif","N",tn,pn)
    doc.setdefault("controles",{})["schema_deterministe"]={"seuils_dzd":{"LOW":ALERT_MIN_DZD,"MEDIUM":ALERT_MEDIUM_DZD,"HIGH":ALERT_HIGH_DZD},
        "details":out,"alerts":[x for x in out if x["level"]!="OK"],"nb_alerts":sum(x["level"]!="OK" for x in out)}
    return doc

# Enrichit les JSON disponibles sans refaire Qwen.
for jf in JSON_DIR.rglob("*.json"):
    try:
        d=json.loads(jf.read_text(encoding="utf-8")); run_document_controls(d); jf.write_text(json.dumps(d,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
    except Exception as e: log("CONTROLE JSON "+str(jf)+" — "+str(e))
print("✅ Contrôles déterministes ajoutés aux JSON")


In [ ]:
# CELLULE 13 — MAPPING EXCEL ANALYSTE V20 (VERIFIE SUR LE MODELE)
CFG_V20 = {'ACTIF_GROUPES': {'N': ['P', 'Q', 'R'], 'N-1': ['M', 'N', 'O'], 'N-2': ['J', 'K', 'L'], 'N-3': ['G', 'H', 'I'], 'N-4': ['D', 'E', 'F']},
 'PASSIF_COLS': {'N': 'H', 'N-1': 'G', 'N-2': 'F', 'N-3': 'E', 'N-4': 'D'},
 'TCR_PAIRES': {'N': ['M', 'N'], 'N-1': ['K', 'L'], 'N-2': ['I', 'J'], 'N-3': ['G', 'H'], 'N-4': ['E', 'F']},
 'ACTIF_ROWS': {'ecarts_acquisition_goodwill': 10,
                'immobilisations_incorporelles': 11,
                'immobilisations_corporelles': 12,
                'terrains': 13,
                'batiments': 14,
                'autres_immobilisations_corporelles': 15,
                'immobilisations_en_concession': 16,
                'immobilisations_en_cours': 17,
                'immobilisations_financieres': 18,
                'titres_mis_en_equivalence': 19,
                'autres_participations_creances': 20,
                'autres_titres_immobilises': 21,
                'prets_actifs_financiers_non_courants': 22,
                'impots_differes_actif': 23,
                'total_actif_non_courant': 24,
                'stocks_encours': 26,
                'creances_emplois_assimiles': 27,
                'clients': 28,
                'autres_debiteurs': 29,
                'impots_assimiles_actif': 30,
                'autres_creances_assimiles': 31,
                'disponibilites_assimiles': 32,
                'placements_financiers_courants': 33,
                'tresorerie_actif': 34,
                'total_actif_courant': 35,
                'total_general_actif': 36},
 'PASSIF_ROWS': {'capital_emis': 6,
                 'capital_non_appele': 7,
                 'primes_reserves': 8,
                 'ecart_reevaluation': 9,
                 'ecart_equivalence': 10,
                 'resultat_net_passif': 11,
                 'report_a_nouveau': 12,
                 'part_societe_consolidante': 13,
                 'part_minoritaires': 14,
                 'total_capitaux_propres': 15,
                 'emprunts_dettes_financieres': 17,
                 'impots_differes_provisionnes': 18,
                 'autres_dettes_non_courantes': 19,
                 'provisions_produits_avance': 20,
                 'total_passifs_non_courants': 21,
                 'fournisseurs_rattaches': 23,
                 'impots_passif': 24,
                 'autres_dettes': 25,
                 'tresorerie_passif': 26,
                 'total_passifs_courants': 27,
                 'total_general_passif': 28},
 'TCR_ROWS': {'ventes_marchandises': 6,
              'produits_fabriques': 7,
              'prestations_services': 8,
              'ventes_travaux': 9,
              'produits_annexes': 10,
              'rabais_remises_ristournes_accordes': 11,
              'chiffre_affaires_net': 12,
              'production_stockee_destockee': 13,
              'production_immobilisee': 14,
              'subvention_exploitation': 15,
              'production_exercice': 16,
              'achats_marchandises_vendues': 17,
              'matieres_premieres': 18,
              'autres_approvisionnements': 19,
              'variation_stocks': 20,
              'achats_etudes_prestations': 21,
              'autres_consommations': 22,
              'rabais_remises_ristournes_obtenus_achats': 23,
              'sous_traitance_generale': 24,
              'locations': 25,
              'entretien_reparations': 26,
              'primes_assurances': 27,
              'personnel_exterieur': 28,
              'remuneration_intermediaires': 29,
              'publicite': 30,
              'deplacements_missions': 31,
              'autres_services': 32,
              'rabais_remises_ristournes_obtenus_services': 33,
              'consommations_exercice': 34,
              'valeur_ajoutee_exploitation': 35,
              'charges_personnel': 36,
              'impots_taxes_assimiles': 37,
              'excedent_brut_exploitation': 38,
              'autres_produits_operationnels': 39,
              'autres_charges_operationnelles': 40,
              'dotations_amortissements': 41,
              'provisions': 42,
              'pertes_valeur': 43,
              'reprises_pertes_valeur_provisions': 44,
              'resultat_operationnel': 45,
              'produits_financiers': 46,
              'charges_financieres': 47,
              'resultat_financier': 48,
              'resultat_ordinaire': 49,
              'elements_extraordinaires_produits': 50,
              'elements_extraordinaires_charges': 51,
              'resultat_extraordinaire': 52,
              'impots_exigibles_resultats': 53,
              'impots_differes_resultats': 54,
              'resultat_net_exercice': 55},
 'ACTIF_FORMULES': [9, 12, 18, 24, 25, 27, 32, 35, 36],
 'PASSIF_FORMULES': [5, 15, 16, 21, 22, 27, 28],
 'TCR_FORMULES': [12, 16, 34, 35, 38, 45, 48, 49, 52, 55]}
FEUILLE_ACTIF="Saisie actif"; FEUILLE_PASSIF="Saisie passif"; FEUILLE_TCR="Saisie TCR"
RANKS=["N","N-1","N-2","N-3","N-4"]

# Résolution modèle / recalcul dans Domino, avec fallback /mnt/data pour test local.
def resolve_model():
    cands=[Path("/mnt/Risk/Model/MODEL ETAT FINANCIER - ANALYSTE COPILOTE V20.xlsx"),Path("/mnt/data/MODEL ETAT FINANCIER - ANALYSTE COPILOTE V20.xlsx")]
    for p in cands:
        if p.exists(): return p
    more=list(Path("/mnt/Risk/Model").glob("*ANALYSTE*V20*.xlsx")) if Path("/mnt/Risk/Model").exists() else []
    return more[0] if more else cands[0]

def resolve_recalc():
    for p in [Path("/mnt/Risk/Model/recalc_liasse.py"),Path("/mnt/data/recalc_liasse.py")]:
        if p.exists(): return p
    return Path("/mnt/Risk/Model/recalc_liasse.py")

MODEL_XLSX=resolve_model(); RECALC_TOOL=resolve_recalc()
print("Modèle Excel:",MODEL_XLSX)
print("Recalc tool:",RECALC_TOOL,"| soffice:",shutil.which("soffice"))


In [ ]:
# CELLULE 14 — CONSTRUCTION EXCEL + IDENTITE + INVENTAIRE COMPLET DES FORMULES FINANCIERES

def is_formula(cell): return isinstance(cell.value,str) and cell.value.startswith("=")
def kdzd(v): return None if v is None else float(v)/1000.0

def table_rows(doc,table):
    out={}; pages=[]
    for p in doc.get("pages",[]):
        if table not in (p.get("classification") or {}).get("types",[]): continue
        rows=(p.get("donnees") or {}).get(table)
        if isinstance(rows,dict):
            for rc,vals in rows.items(): out.setdefault(rc,{}).update({k:v for k,v in vals.items() if v is not None})
            pages.append(p.get("numero_page_scannee"))
    return out,[x for x in pages if x]

def table_page_records(doc,table):
    out=[]
    for p in doc.get("pages",[]):
        if table not in (p.get("classification") or {}).get("types",[]): continue
        e=normalize_page_identity(p.get("entete_page") or {})
        out.append({"page":p.get("numero_page_scannee"),"entreprise":e.get("entreprise"),"nif_15":e.get("nif_15"),
                    "identity_source":e.get("identity_source"),"document":(doc.get("document") or {}).get("fichier_source")})
    return out

def source_text(doc,table,period,year):
    _,pages=table_rows(doc,table); f=(doc.get("document") or {}).get("fichier_source") or "?"
    pp="p."+",".join(map(str,pages)) if pages else "page ?"
    return f"{year} — {f} — {pp}" + (" — colonne N-1" if period=="N1" else "")

def doc_year(doc):
    y=(doc.get("identite") or {}).get("annee")
    if y: return int(y)
    return parse_year((doc.get("identite") or {}).get("exercice_au")) or parse_year((doc.get("document") or {}).get("fichier_source"))

def choose_candidates(docs):
    # Choix par (année, tableau), priorité au N de son propre PDF, puis au N-1 du PDF suivant.
    cands=defaultdict(list)
    for d in docs:
        y=doc_year(d)
        if not y: continue
        for table in ("ACTIF","PASSIF","TCR"):
            rows,_=table_rows(d,table)
            if rows: cands[(y,table)].append((0,d,"N"))
            # Fallback N-1 uniquement PASSIF/TCR : le formulaire ACTIF N-1 ne fournit que le Net.
            if table in {"PASSIF","TCR"} and rows and any(any(k.startswith("n1") for k in vals) for vals in rows.values()):
                cands[(y-1,table)].append((1,d,"N1"))
    return {k:sorted(v,key=lambda x:x[0])[0] for k,v in cands.items()}

def majority_identity(docs):
    names=[]; nifs=[]
    for d in docs:
        e=d.get("identite") or {}
        if e.get("entreprise"): names.append(e.get("entreprise"))
        if e.get("nif_15"): nifs.append(e.get("nif_15"))
    return {"entreprise":_name_medoid(names) or "","nif_15":Counter(nifs).most_common(1)[0][0] if nifs else ""}

def reference_identity_from_candidates(candidates, latest_year):
    records=[]
    for table in ("ACTIF","PASSIF","TCR"):
        cand=candidates.get((latest_year,table))
        if not cand: continue
        _,doc,_=cand; records.extend(table_page_records(doc,table))
    names=[r["entreprise"] for r in records if r.get("entreprise")]
    nifs=[r["nif_15"] for r in records if r.get("nif_15")]
    return {"year":latest_year,"entreprise":_name_medoid(names),"nif_15":Counter(nifs).most_common(1)[0][0] if nifs else None,"records":records}

def audit_table_identity(doc,table,year,reference):
    recs=table_page_records(doc,table)
    names=[r.get("entreprise") for r in recs if r.get("entreprise")]
    nifs=[r.get("nif_15") for r in recs if r.get("nif_15")]
    actual_name=_name_medoid(names)
    actual_nif=Counter(nifs).most_common(1)[0][0] if nifs else None
    ref_name=reference.get("entreprise"); ref_nif=reference.get("nif_15")
    scores=[company_similarity(r.get("entreprise"),ref_name) for r in recs if r.get("entreprise") and ref_name]
    min_score=min(scores) if scores else None
    name_missing=not names or not ref_name
    nif_missing=not nifs or not ref_nif
    bad_name_pages=[r for r in recs if ref_name and r.get("entreprise") and (company_similarity(r.get("entreprise"),ref_name) or 0)<IDENTITY_SIMILARITY_MIN]
    bad_nif_pages=[r for r in recs if ref_nif and r.get("nif_15") and r.get("nif_15")!=ref_nif]
    missing_pages=[r for r in recs if not r.get("entreprise") or not r.get("nif_15")]

    # Incohérence interne entre pages d'un même tableau (notamment TCR P1/P2).
    internal_name_bad=False
    for i in range(len(names)):
        for j in range(i+1,len(names)):
            if (company_similarity(names[i],names[j]) or 0)<IDENTITY_SIMILARITY_MIN: internal_name_bad=True
    internal_nif_bad=len(set(nifs))>1

    if bad_nif_pages or bad_name_pages or internal_name_bad or internal_nif_bad:
        level="HIGH"; status="MISMATCH"
    elif name_missing or nif_missing or missing_pages:
        level="MEDIUM"; status="INCOMPLET"
    else:
        level="OK"; status="OK"
    return {"year":year,"table":table,"status":status,"level":level,"entreprise":actual_name,"nif_15":actual_nif,
            "reference_year":reference.get("year"),"reference_entreprise":ref_name,"reference_nif_15":ref_nif,
            "similarity_min":None if min_score is None else round(min_score,4),"pages":recs,
            "bad_name_pages":bad_name_pages,"bad_nif_pages":bad_nif_pages,"missing_pages":missing_pages,
            "internal_name_bad":internal_name_bad,"internal_nif_bad":internal_nif_bad}

def build_identity_audits(candidates, years, reference):
    out={}
    for (year,table),(prio,doc,period) in candidates.items():
        if year in years: out[(year,table)]=audit_table_identity(doc,table,year,reference)
    return out

def clear_writable_zones(wb):
    for rank in RANKS:
        ga=CFG_V20["ACTIF_GROUPES"][rank]; cp=CFG_V20["PASSIF_COLS"][rank]; gt=CFG_V20["TCR_PAIRES"][rank]
        for rw in CFG_V20["ACTIF_ROWS"].values():
            for c in ga:
                cell=wb[FEUILLE_ACTIF][f"{c}{rw}"]
                if not is_formula(cell): cell.value=None; cell.comment=None
        for rw in CFG_V20["PASSIF_ROWS"].values():
            cell=wb[FEUILLE_PASSIF][f"{cp}{rw}"]
            if not is_formula(cell): cell.value=None; cell.comment=None
        for rw in CFG_V20["TCR_ROWS"].values():
            for c in gt:
                cell=wb[FEUILLE_TCR][f"{c}{rw}"]
                if not is_formula(cell): cell.value=None; cell.comment=None

def collect_formula_control(controls,cell,observed,meta):
    if observed is None: return
    rec=dict(meta); rec.update({"sheet":cell.parent.title,"cell":cell.coordinate,"observed_kdzd":kdzd(observed)})
    controls.append(rec)

def write_one_table(wb,table,rank,doc,period,year,controls):
    rows,_=table_rows(doc,table); written=0
    src=source_text(doc,table,period,year)
    if table=="ACTIF":
        cols=CFG_V20["ACTIF_GROUPES"][rank]
        mapping=[("montant_brut",cols[0]),("amortissements_provisions_pertes",cols[1]),("net_n",cols[2])] if period=="N" else [("net_n1",cols[2])]
        for rc,rw in CFG_V20["ACTIF_ROWS"].items():
            vals=rows.get(rc,{})
            for field,col in mapping:
                v=vals.get(field)
                if v is None: continue
                cell=wb[FEUILLE_ACTIF][f"{col}{rw}"]
                if is_formula(cell): collect_formula_control(controls,cell,v,{"table":"ACTIF","row":rc,"column":field,"source":src,"year":year})
                else: cell.value=kdzd(v); cell.number_format='#,##0'; written+=1
    elif table=="PASSIF":
        col=CFG_V20["PASSIF_COLS"][rank]; field="n" if period=="N" else "n1"
        for rc,rw in CFG_V20["PASSIF_ROWS"].items():
            v=rows.get(rc,{}).get(field)
            if v is None: continue
            cell=wb[FEUILLE_PASSIF][f"{col}{rw}"]
            if is_formula(cell): collect_formula_control(controls,cell,v,{"table":"PASSIF","row":rc,"column":field,"source":src,"year":year})
            else: cell.value=kdzd(v); cell.number_format='#,##0'; written+=1
    elif table=="TCR":
        dcol,ccol=CFG_V20["TCR_PAIRES"][rank]; df="n_debit" if period=="N" else "n1_debit"; cf="n_credit" if period=="N" else "n1_credit"
        for rc,rw in CFG_V20["TCR_ROWS"].items():
            vals=rows.get(rc,{})
            for field,col in ((df,dcol),(cf,ccol)):
                v=vals.get(field)
                if v is None: continue
                cell=wb[FEUILLE_TCR][f"{col}{rw}"]
                if is_formula(cell): collect_formula_control(controls,cell,v,{"table":"TCR","row":rc,"column":field,"source":src,"year":year})
                else: cell.value=kdzd(v); cell.number_format='#,##0'; written+=1
    return written,src

# Alias de lignes formule qui reprennent un total imprimé ailleurs dans la liasse.
# Cela permet de contrôler AUSSI les formules de tête du modèle sans inventer de donnée Qwen.
ACTIF_FORMULA_ROW_ALIASES = {
    9: "total_actif_non_courant",
    25: "total_actif_courant",
}
PASSIF_FORMULA_ROW_ALIASES = {
    5: "total_capitaux_propres",
    16: "total_passifs_non_courants",
    22: "total_passifs_courants",
}

def _formula_control_record(cell,table,row_code,column_id,observed_dzd,source,year):
    return {
        "sheet":cell.parent.title, "cell":cell.coordinate, "table":table, "row":row_code, "column":column_id,
        "year":year, "source":source, "formula":cell.value,
        "observed_dzd":None if observed_dzd is None else float(observed_dzd),
        "observed_kdzd":None if observed_dzd is None else kdzd(observed_dzd),
    }

def build_all_financial_formula_controls(wb,candidates,year_to_rank):
    """Inventorie toutes les formules financières comparables au PDF/Qwen.

    On ne dépend plus de write_one_table() pour décider quelles formules contrôler.
    Chaque cellule formule de la zone financière ACTIF/PASSIF/TCR est parcourue et reliée
    à un champ canonique de la liasse. Les formules techniques PASSIF 29/30 sont exclues
    car elles n'ont pas de valeur imprimée équivalente à comparer à Qwen.
    """
    controls=[]
    actif_row_to_rc={rw:rc for rc,rw in CFG_V20["ACTIF_ROWS"].items()}
    actif_row_to_rc.update(ACTIF_FORMULA_ROW_ALIASES)
    passif_row_to_rc={rw:rc for rc,rw in CFG_V20["PASSIF_ROWS"].items()}
    passif_row_to_rc.update(PASSIF_FORMULA_ROW_ALIASES)
    tcr_row_to_rc={rw:rc for rc,rw in CFG_V20["TCR_ROWS"].items()}

    for (year,table),(prio,doc,period) in candidates.items():
        rank=year_to_rank.get(year)
        if not rank: continue
        rows,_=table_rows(doc,table)
        src=source_text(doc,table,period,year)

        if table=="ACTIF":
            cols=CFG_V20["ACTIF_GROUPES"][rank]
            field_by_col={cols[0]:"montant_brut",cols[1]:"amortissements_provisions_pertes",cols[2]:"net_n"}
            for rownum,rc in sorted(actif_row_to_rc.items()):
                vals=rows.get(rc,{})
                for col,field in field_by_col.items():
                    cell=wb[FEUILLE_ACTIF][f"{col}{rownum}"]
                    if not is_formula(cell): continue
                    controls.append(_formula_control_record(cell,"ACTIF",rc,field,vals.get(field),src,year))

        elif table=="PASSIF":
            col=CFG_V20["PASSIF_COLS"][rank]
            field="n" if period=="N" else "n1"
            for rownum,rc in sorted(passif_row_to_rc.items()):
                cell=wb[FEUILLE_PASSIF][f"{col}{rownum}"]
                if not is_formula(cell): continue
                controls.append(_formula_control_record(cell,"PASSIF",rc,field,rows.get(rc,{}).get(field),src,year))

        elif table=="TCR":
            dcol,ccol=CFG_V20["TCR_PAIRES"][rank]
            df="n_debit" if period=="N" else "n1_debit"
            cf="n_credit" if period=="N" else "n1_credit"
            for rownum,rc in sorted(tcr_row_to_rc.items()):
                vals=rows.get(rc,{})
                for col,field in ((dcol,df),(ccol,cf)):
                    cell=wb[FEUILLE_TCR][f"{col}{rownum}"]
                    if not is_formula(cell): continue
                    controls.append(_formula_control_record(cell,"TCR",rc,field,vals.get(field),src,year))

    # dédoublonnage défensif par cellule/exercice/table
    uniq={}
    for rec in controls: uniq[(rec["sheet"],rec["cell"],rec["year"],rec["table"])]=rec
    return list(uniq.values())

def count_formula_controls_by_table(controls):
    return dict(Counter(x.get("table") for x in controls))

def hide_missing_years(wb,year_to_rank,present_years):
    present_ranks={year_to_rank[y] for y in present_years if y in year_to_rank}
    for rank in RANKS:
        hide=rank not in present_ranks
        for c in CFG_V20["ACTIF_GROUPES"][rank]: wb[FEUILLE_ACTIF].column_dimensions[c].hidden=hide
        wb[FEUILLE_PASSIF].column_dimensions[CFG_V20["PASSIF_COLS"][rank]].hidden=hide
        for c in CFG_V20["TCR_PAIRES"][rank]: wb[FEUILLE_TCR].column_dimensions[c].hidden=hide

def write_sources(wb,year_to_rank,sources):
    # source sous chaque exercice, séparée par état
    for year,rank in year_to_rank.items():
        for table,ws,col in [
            ("ACTIF",wb[FEUILLE_ACTIF],CFG_V20["ACTIF_GROUPES"][rank][0]),
            ("PASSIF",wb[FEUILLE_PASSIF],CFG_V20["PASSIF_COLS"][rank]),
            ("TCR",wb[FEUILLE_TCR],CFG_V20["TCR_PAIRES"][rank][0])]:
            cell=ws[f"{col}3"]; cell.value=sources.get((year,table),""); cell.alignment=Alignment(wrap_text=True,vertical="top"); cell.font=Font(size=8,italic=True)
    for ws in (wb[FEUILLE_ACTIF],wb[FEUILLE_PASSIF],wb[FEUILLE_TCR]): ws.row_dimensions[3].height=max(ws.row_dimensions[3].height or 15,30)

def _identity_group_columns(table,rank):
    if table=="ACTIF": return list(CFG_V20["ACTIF_GROUPES"][rank])
    if table=="PASSIF": return [CFG_V20["PASSIF_COLS"][rank]]
    return list(CFG_V20["TCR_PAIRES"][rank])

def _identity_alert_text(audit):
    sim=audit.get("similarity_min")
    simtxt="non calculable" if sim is None else f"{sim:.2%}"
    pages=", ".join(f"{r.get('document')} p.{r.get('page')}" for r in audit.get("pages",[])) or "?"
    return ("CONTROLE IDENTITE YA-RISK\n\n"
            f"Statut : {audit.get('status')} / {audit.get('level')}\n"
            f"Exercice : {audit.get('year')} | Tableau : {audit.get('table')}\n"
            f"Raison sociale lue : {audit.get('entreprise') or 'NON LUE'}\n"
            f"NIF lu : {audit.get('nif_15') or 'NON LU'}\n"
            f"Référence {audit.get('reference_year')} : {audit.get('reference_entreprise') or 'NON LUE'}\n"
            f"NIF référence : {audit.get('reference_nif_15') or 'NON LU'}\n"
            f"Similarité minimale : {simtxt} (seuil {IDENTITY_SIMILARITY_MIN:.0%})\n"
            f"Pages : {pages}")

def apply_identity_display(wb,year_to_rank,reference,audits):
    """Le dernier exercice est visible. Les exercices comparés conformes sont masqués (cellules vides).
    Toute divergence/missing reste visible et colorée sur ACTIF/PASSIF/TCR.
    """
    ref_year=reference.get("year")
    for year,rank in year_to_rank.items():
        for table,ws in [("ACTIF",wb[FEUILLE_ACTIF]),("PASSIF",wb[FEUILLE_PASSIF]),("TCR",wb[FEUILLE_TCR])]:
            audit=audits.get((year,table))
            cols=_identity_group_columns(table,rank)
            lead=cols[0]
            # nettoyer uniquement la zone d'identité de cet exercice
            for row in (1,2):
                for col in cols:
                    c=ws[f"{col}{row}"]; c.fill=PatternFill(fill_type=None); c.comment=None
            if not audit:
                continue
            is_ref=(year==ref_year)
            conform=(audit.get("level")=="OK")
            if (not is_ref) and conform:
                ws[f"{lead}1"]=""; ws[f"{lead}2"]=""
                continue
            name=audit.get("entreprise") or (reference.get("entreprise") if is_ref else "NON LU")
            nif=audit.get("nif_15") or (reference.get("nif_15") if is_ref else "NON LU")
            ws[f"{lead}1"]=name or ""; ws[f"{lead}2"].number_format='@'; ws[f"{lead}2"]=nif or ""
            for row in (1,2):
                ws[f"{lead}{row}"].alignment=Alignment(wrap_text=True,vertical="top")
            if audit.get("level") in {"MEDIUM","HIGH"}:
                fill=FILL_MEDIUM if audit.get("level")=="MEDIUM" else FILL_HIGH
                color=COLOR_MEDIUM if audit.get("level")=="MEDIUM" else COLOR_HIGH
                txt=_identity_alert_text(audit)
                for row in (1,2):
                    for col in cols:
                        c=ws[f"{col}{row}"]; c.fill=fill; border_alert(c,color,"thick" if audit.get("level")=="HIGH" else "medium")
                ws[f"{lead}1"].comment=Comment(txt,"Ya-Risk")
                ws[f"{lead}2"].comment=Comment(txt,"Ya-Risk")
    for ws in (wb[FEUILLE_ACTIF],wb[FEUILLE_PASSIF],wb[FEUILLE_TCR]):
        ws.row_dimensions[1].height=max(ws.row_dimensions[1].height or 15,30)
        ws.row_dimensions[2].height=max(ws.row_dimensions[2].height or 15,24)


In [ ]:
# CELLULE 15 — RECALCUL + AUDIT DE TOUTES LES FORMULES + COMMENTAIRES / POLICE ROUGE
# Ces couleurs restent utilisées pour le contrôle IDENTITE et la feuille de synthèse.
# Les alertes de FORMULE dans ACTIF/PASSIF/TCR ne modifient JAMAIS le fond de la cellule.
FILL_LOW=PatternFill("solid",start_color="FFF2CC")
FILL_MEDIUM=PatternFill("solid",start_color="F8CBAD")
FILL_HIGH=PatternFill("solid",start_color="FFC7CE")
COLOR_LOW="C9A227"; COLOR_MEDIUM="ED7D31"; COLOR_HIGH="FF0000"
FORMULA_ALERT_FONT_RGB="FFFF0000"

def border_alert(cell,color,style="medium"):
    # Réservé aux alertes IDENTITE. Les écarts de formule n'utilisent ni bordure ni remplissage.
    s=Side(style=style,color=color); cell.border=Border(left=s,right=s,top=s,bottom=s)

def recalc_excel(path,timeout=60):
    """Recalcule les formules AVANT les commentaires/polices d'alerte.
    Aucun passage LibreOffice ne doit avoir lieu après apply_formula_alerts().
    """
    path=Path(path)
    soffice=shutil.which("soffice") or shutil.which("libreoffice")
    if soffice:
        try:
            with tempfile.TemporaryDirectory(prefix="yarisk_lo_") as td:
                td=Path(td); outdir=td/"out"; outdir.mkdir(); profile=td/"profile"
                cmd=[soffice,"--headless","--norestore",f"-env:UserInstallation={profile.as_uri()}",
                     "--convert-to","xlsx","--outdir",str(outdir),str(path.resolve())]
                r=subprocess.run(cmd,capture_output=True,text=True,timeout=timeout)
                converted=outdir/path.name
                if r.returncode==0 and converted.exists():
                    shutil.copy2(converted,path)
                    return {"ok":True,"code":0,"message":"OK LibreOffice fast convert","stdout":(r.stdout or "")[-500:]}
        except Exception as e:
            fast_error=type(e).__name__+": "+str(e)
        else:
            fast_error="conversion LibreOffice sans fichier de sortie"
    else:
        fast_error="soffice indisponible"
    if RECALC_TOOL.exists() and soffice:
        try:
            r=subprocess.run([sys.executable,str(RECALC_TOOL),str(path),str(min(timeout,90))],capture_output=True,text=True,timeout=min(timeout,90)+30)
            return {"ok":r.returncode==0,"code":r.returncode,"message":"OK fallback recalc_liasse.py" if r.returncode==0 else (r.stderr or r.stdout)[-1000:],"fast_error":fast_error}
        except Exception as e:
            return {"ok":False,"message":fast_error+" | fallback "+type(e).__name__+": "+str(e)}
    return {"ok":False,"message":fast_error}

def scan_bad_formulas(wb):
    out=[]
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                if isinstance(c.value,str) and c.value.startswith("=") and "#REF!" in c.value.upper(): out.append(f"{ws.title}!{c.coordinate} = {c.value}")
    return out

def evaluate_formula_cells(path,controls):
    """Compare CHAQUE formule financière inventoriée à la valeur PDF/Qwen correspondante.

    - valeur Qwen présente + formule numérique : comparaison DZD avec LOW/MEDIUM/HIGH ;
    - valeur Qwen absente + formule = 0 : pas d'alerte ;
    - valeur Qwen absente + formule non nulle : alerte explicite QWEN_ABSENT ;
    - formule non calculée/non numérique : alerte technique explicite si une valeur Qwen existe.
    """
    wbv=load_workbook(path,data_only=True); results=[]
    for ctrl in controls:
        observed=ctrl.get("observed_kdzd")
        calc=wbv[ctrl["sheet"]][ctrl["cell"]].value
        rec=dict(ctrl); rec["calculated_kdzd"]=float(calc) if isinstance(calc,(int,float)) else calc

        if not isinstance(calc,(int,float)):
            if observed is None:
                rec.update({"status":"NON_COMPARABLE","level":"OK","difference_dzd":None})
            else:
                rec.update({"status":"FORMULE_NON_CALCULEE","level":"HIGH","difference_dzd":None})
            results.append(rec); continue

        calc=float(calc)
        if observed is None:
            diff_dzd=calc*1000.0
            if abs(diff_dzd) < ALERT_MIN_DZD:
                rec.update({"status":"OK_VIDE","level":"OK","difference_kdzd":calc,"difference_dzd":round(diff_dzd,2)})
            else:
                rec.update({"status":"QWEN_ABSENT","level":ecart_level(diff_dzd),"difference_kdzd":calc,"difference_dzd":round(diff_dzd,2)})
            results.append(rec); continue

        diff_kdzd=calc-float(observed); diff_dzd=diff_kdzd*1000.0; level=ecart_level(diff_dzd)
        rec.update({"difference_kdzd":round(diff_kdzd,6),"difference_dzd":round(diff_dzd,2),"level":level,
                    "status":"OK" if level=="OK" else "ECART_FORMULE"})
        results.append(rec)
    return results

def formula_alert_comment(rec):
    observed=rec.get("observed_kdzd"); calc=rec.get("calculated_kdzd"); diff_dzd=rec.get("difference_dzd")
    level=rec.get("level"); status=rec.get("status")
    obs_txt="NON EXTRAITE / VIDE" if observed is None else f"{float(observed):,.3f} KDZD"
    calc_txt="NON CALCULEE" if not isinstance(calc,(int,float)) else f"{float(calc):,.3f} KDZD"
    diff_txt="NON CALCULABLE" if diff_dzd is None else f"{float(diff_dzd):+,.0f} DZD"
    pct=None
    if isinstance(observed,(int,float)) and abs(float(observed))>1e-12 and isinstance(calc,(int,float)):
        pct=100.0*(float(calc)-float(observed))/abs(float(observed))
    formula=rec.get("formula") or "?"
    return ("⚠ YA-RISK — CONTROLE FORMULE / PDF-QWEN\n\n"
            f"Statut : {status}\nNiveau : {level}\n"
            f"Tableau : {rec.get('table')}\nPoste : {rec.get('row')}\nCase métier : {rec.get('column')}\n"
            f"Exercice : {rec.get('year')}\nCellule Excel : {rec.get('sheet')}!{rec.get('cell')}\n\n"
            f"Valeur PDF/Qwen : {obs_txt}\nValeur calculée : {calc_txt}\nEcart : {diff_txt}\n" +
            (f"Ecart relatif : {pct:+.4f}%\n" if pct is not None else "") +
            f"Formule Excel : {formula}\n\nSource : {rec.get('source','')}\n\n"
            "La valeur Qwen n'est jamais remplacée automatiquement. Vérification analyste requise.")

def _set_formula_alert_font_red(cell):
    # Conserve taille, gras, italique, police, etc. ; seule la couleur des chiffres devient rouge.
    f=copy.copy(cell.font); f.color=FORMULA_ALERT_FONT_RGB; cell.font=f

def fill_signature(cell):
    f=cell.fill
    return (f.fill_type, getattr(f.fgColor,"type",None), getattr(f.fgColor,"rgb",None), getattr(f.fgColor,"indexed",None),
            getattr(f.bgColor,"type",None), getattr(f.bgColor,"rgb",None), getattr(f.bgColor,"indexed",None))

def font_is_formula_alert_red(cell):
    c=cell.font.color
    if c is None: return False
    if getattr(c,"type",None)=="rgb":
        return str(getattr(c,"rgb","") or "").upper().endswith("FF0000")
    return False

def apply_formula_alerts(wb,formula_results):
    """Après le dernier recalcul : commentaire + chiffres rouges, SANS TOUCHER au fond ni à la bordure."""
    applied=[]
    for rec in formula_results:
        level=rec.get("level")
        if level not in ("LOW","MEDIUM","HIGH"): continue
        cell=wb[rec["sheet"]][rec["cell"]]
        before_fill=fill_signature(cell)
        com=Comment(formula_alert_comment(rec),"Ya-Risk")
        try: com.width=520; com.height=300
        except Exception: pass
        cell.comment=com
        _set_formula_alert_font_red(cell)
        after_fill=fill_signature(cell)
        rec["fill_unchanged_in_memory"]=(before_fill==after_fill)
        applied.append((rec["sheet"],rec["cell"]))
    return applied

def add_control_sheet_to_wb(wb,client,reference,years,sources,formula_results,cross_doc_alerts,identity_audits,bad_formulas,recalc_info):
    name="0.Controles YaRisk V26"
    if name in wb.sheetnames: del wb[name]
    ws=wb.create_sheet(name,0); ws["A1"]="YA-RISK — CONTROLES PRE-CERTIFICATION V26"; ws["A1"].font=Font(bold=True,size=14)
    rows=[["Client",client],["Exercice de référence",reference.get("year")],["Raison sociale référence",reference.get("entreprise")],["NIF référence",reference.get("nif_15")],
          ["Seuil similarité raison sociale",f"{IDENTITY_SIMILARITY_MIN:.0%}"],["Exercices",", ".join(map(str,years))],
          ["Seuil LOW à partir de (DZD)",ALERT_MIN_DZD],["MEDIUM à partir de (DZD)",ALERT_MEDIUM_DZD],["HIGH à partir de (DZD)",ALERT_HIGH_DZD],
          ["Présentation écart formule","Commentaire cellule + chiffres rouges ; fond inchangé"],["Recalcul LibreOffice",recalc_info.get("message")]]
    for r in rows: ws.append(r)
    ws.append([]); ws.append(["CONTROLES IDENTITE","Niveau","Exercice","Tableau","Raison sociale lue","NIF lu","Similarité min","Raison sociale référence","NIF référence","Pages"])
    for (year,table),a in sorted(identity_audits.items(),reverse=True):
        pages="; ".join(f"{r.get('document')} p.{r.get('page')}" for r in a.get("pages",[]))
        ws.append([a.get("status"),a.get("level"),year,table,a.get("entreprise"),a.get("nif_15"),a.get("similarity_min"),a.get("reference_entreprise"),a.get("reference_nif_15"),pages])
        if a.get("level")=="HIGH":
            for c in ws[ws.max_row]: c.fill=FILL_HIGH
        elif a.get("level")=="MEDIUM":
            for c in ws[ws.max_row]: c.fill=FILL_MEDIUM
    ws.append([]); ws.append(["ECARTS FORMULES", "Niveau","Exercice","Tableau","Poste","Case","PDF/Qwen KDZD","Formule KDZD","Ecart DZD","Cellule","Formule Excel","Source"])
    for x in formula_results:
        if x.get("level") in ("LOW","MEDIUM","HIGH") or x.get("status")=="FORMULE_NON_CALCULEE":
            ws.append([x.get("status"),x.get("level"),x.get("year"),x.get("table"),x.get("row"),x.get("column"),x.get("observed_kdzd"),x.get("calculated_kdzd"),x.get("difference_dzd"),x.get("cell"),x.get("formula"),x.get("source")])
            # Couleur dans la feuille de synthèse uniquement ; les feuilles métier gardent leur fond d'origine.
            if x.get("level")=="LOW":
                for c in ws[ws.max_row]: c.fill=FILL_LOW
            elif x.get("level")=="MEDIUM":
                for c in ws[ws.max_row]: c.fill=FILL_MEDIUM
            elif x.get("level")=="HIGH":
                for c in ws[ws.max_row]: c.fill=FILL_HIGH
    ws.append([]); ws.append(["ECARTS INTER-DOCUMENTS","Niveau","Année","Field","Valeur 1 DZD","Valeur 2 DZD","Ecart DZD","Source 1","Source 2"])
    for x in cross_doc_alerts:
        ws.append(["ALERTE",x.get("level"),x.get("year"),x.get("field_key"),x.get("value1"),x.get("value2"),x.get("difference_dzd"),x.get("source1"),x.get("source2")])
    ws.append([]); ws.append(["ANOMALIES MODELE EXCEL"])
    for x in bad_formulas: ws.append([x])
    ws.freeze_panes="A2"
    for c,w in {"A":28,"B":14,"C":12,"D":16,"E":32,"F":22,"G":18,"H":18,"I":18,"J":18,"K":48,"L":52}.items(): ws.column_dimensions[c].width=w
    for row in ws.iter_rows():
        for c in row: c.alignment=Alignment(wrap_text=True,vertical="top")


In [ ]:
# CELLULE 16 — CONTROLE INTER-DOCUMENTS + POST-TRAITEMENT CLIENT V26
def canonical_year_observations(docs):
    # clé = (année, table, row, côté/colonne). On conserve toutes les sources pour comparer N vs N-1.
    bag=defaultdict(list)
    for d in docs:
        y=doc_year(d)
        if not y: continue
        f=(d.get("document") or {}).get("fichier_source") or "?"
        for p in d.get("pages",[]):
            page=p.get("numero_page_scannee")
            for table,rows in (p.get("donnees") or {}).items():
                if table not in ("ACTIF","PASSIF","TCR") or not isinstance(rows,dict): continue
                for rc,vals in rows.items():
                    for col,v in vals.items():
                        if v is None: continue
                        if table=="ACTIF":
                            if col=="net_n1": yy=y-1; keycol="net"
                            elif col in ("montant_brut","amortissements_provisions_pertes","net_n"): yy=y; keycol=col.replace("net_n","net")
                            else: continue
                        elif table=="PASSIF": yy=y if col=="n" else y-1 if col=="n1" else None; keycol="value"
                        else:
                            if col.startswith("n1_"): yy=y-1; keycol=col[3:]
                            elif col.startswith("n_"): yy=y; keycol=col[2:]
                            else: continue
                        if yy is not None: bag[(yy,table,rc,keycol)].append({"value":float(v),"source":f"{f} — p.{page} — {col}"})
    return bag

def cross_document_alerts(docs):
    out=[]
    for (year,table,rc,col),obs in canonical_year_observations(docs).items():
        if len(obs)<2: continue
        # compare chaque observation à la première; aucune écrasée.
        base=obs[0]
        for other in obs[1:]:
            diff=other["value"]-base["value"]; level=ecart_level(diff)
            if level!="OK": out.append({"year":year,"field_key":f"{table}.{rc}.{col}","value1":base["value"],"value2":other["value"],"difference_dzd":round(diff,2),"level":level,"source1":base["source"],"source2":other["source"]})
    return out

def build_client_excel(client,docs):
    if not MODEL_XLSX.exists(): raise FileNotFoundError("Modèle Excel introuvable: "+str(MODEL_XLSX))
    candidates=choose_candidates(docs); years=sorted({y for (y,t) in candidates},reverse=True)
    if not years: raise ValueError("Aucun exercice détecté pour "+client)
    latest=max(years); year_to_rank={latest-i:RANKS[i] for i in range(5)}; years_excel=[y for y in years if y in year_to_rank]
    reference=reference_identity_from_candidates(candidates,latest)
    identity_audits=build_identity_audits(candidates,years_excel,reference)

    out=XLSX_DIR/("PRECERT_V26_"+re.sub(r"[^A-Za-z0-9_.-]+","_",client)+".xlsx")
    shutil.copy2(MODEL_XLSX,out); wb=load_workbook(out); clear_writable_zones(wb)
    bad=scan_bad_formulas(wb); controls=[]; sources={}; written=0
    for (year,table),(prio,doc,period) in candidates.items():
        if year not in year_to_rank: continue
        n,src=write_one_table(wb,table,year_to_rank[year],doc,period,year,controls); written+=n; sources[(year,table)]=src
    write_sources(wb,year_to_rank,sources); hide_missing_years(wb,year_to_rank,years_excel)
    # Inventaire COMPLET des formules financières comparables, indépendamment du chemin d'écriture des données.
    controls=build_all_financial_formula_controls(wb,candidates,year_to_rank)
    log(f"{client} — formules financières contrôlées: {len(controls)} {count_formula_controls_by_table(controls)}")
    try:
        wb.calculation.fullCalcOnLoad=True; wb.calculation.forceFullCalc=True; wb.calculation.calcMode="auto"
    except Exception: pass
    wb.save(out)

    # UNIQUE recalcul final des formules. Les alertes visuelles sont posées APRES et ne sont plus repassées dans LibreOffice.
    recalc=recalc_excel(out)
    formula_results=evaluate_formula_cells(out,controls) if recalc.get("ok") else [dict(x,status="NON_CALCULE",level="NON_CALCULE") for x in controls]
    cross=cross_document_alerts(docs)

    wbf=load_workbook(out)
    expected=[x for x in formula_results if x.get("level") in ("LOW","MEDIUM","HIGH")]
    fills_before={(x["sheet"],x["cell"]):fill_signature(wbf[x["sheet"]][x["cell"]]) for x in expected}
    apply_formula_alerts(wbf,formula_results)
    apply_identity_display(wbf,year_to_rank,reference,identity_audits)
    add_control_sheet_to_wb(wbf,client,reference,years_excel,sources,formula_results,cross,identity_audits,bad,recalc)
    wbf.save(out)

    # Vérification stricte du fichier FINAL : commentaire présent + chiffres rouges + fond inchangé.
    verify=load_workbook(out,read_only=False,data_only=False)
    persisted=0; fills_ok=0
    for x in expected:
        c=verify[x["sheet"]][x["cell"]]
        if c.comment is not None and font_is_formula_alert_red(c): persisted+=1
        if fill_signature(c)==fills_before[(x["sheet"],x["cell"])]: fills_ok+=1
    if persisted != len(expected):
        raise RuntimeError(f"Alertes formule non persistées: {persisted}/{len(expected)}")
    if fills_ok != len(expected):
        raise RuntimeError(f"Fond de cellule modifié sur une alerte formule: {fills_ok}/{len(expected)} inchangés")
    return out,{"client":client,"reference_identity":reference,"identity_audits":{f"{y}.{t}":a for (y,t),a in identity_audits.items()},
                "years":years_excel,"written":written,"sources":{f"{y}.{t}":s for (y,t),s in sources.items()},
                "formula_controls":formula_results,"formula_controls_total":len(formula_results),"formula_controls_by_table":count_formula_controls_by_table(formula_results),
                "formula_alerts_expected":len(expected),"formula_alerts_persisted":persisted,"formula_alert_fills_unchanged":fills_ok,
                "cross_document_alerts":cross,"model_formula_warnings":bad,"recalc":recalc,"final_recalc_after_alerts":False}

clients=defaultdict(list)
for jf in JSON_DIR.rglob("*.json"):
    rel=jf.relative_to(JSON_DIR); client=rel.parts[0] if len(rel.parts)>1 else "_RACINE_"
    try: clients[client].append(json.loads(jf.read_text(encoding="utf-8")))
    except Exception as e: log("JSON illisible "+str(jf)+" — "+str(e))

log(f"V26 post-traitement — {len(clients)} client(s)")
for client,docs in sorted(clients.items()):
    try:
        out,report=build_client_excel(client,docs)
        bundle={"schema_version":SCHEMA_VERSION,"schema_registry_version":SCHEMA_REGISTRY_VERSION,"workflow":{"statut":"A_CERTIFIER","certifie":False,"version_pipeline":"V26_ALL_FORMULAS_REDFONT"},
                "dossier":{"reference":client},"documents":docs,"excel_pre_certification":str(out),"rapport_excel":report}
        jp=CONTROL_DIR/(re.sub(r"[^A-Za-z0-9_.-]+","_",client)+".avant_certification.json")
        jp.write_text(json.dumps(bundle,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
        print("CLIENT",client,"| exercices",report["years"],"| écrits",report["written"],"| alertes formule",report["formula_alerts_expected"],"persistées",report["formula_alerts_persisted"],"| Excel",out)
    except Exception as e:
        log("POST-TRAITEMENT "+client+" — "+type(e).__name__+": "+str(e)); traceback.print_exc()

print("✅ V26 terminé — Excel:",XLSX_DIR)
print("✅ JSON audit:",CONTROL_DIR)


In [ ]:
# CELLULE 17 — TESTS DE COHERENCE V26 (A EXECUTER)
_tests=[]
def check(name,cond,detail=""):
    ok=bool(cond); _tests.append(ok); print(("✅" if ok else "❌"),name,detail if not ok else ""); return ok

# Contrat unique
check("field_id uniques",len(FIELD_REGISTRY)==len(set(FIELD_REGISTRY)))
check("TCR 4 colonnes exactes",TABLE_SCHEMAS["TCR"]["cols"]==["n_debit","n_credit","n1_debit","n1_credit"])
check("TCR P1/P2 couvrent chaque ligne une fois",set(TCR_P1_ROWS).isdisjoint(TCR_P2_ROWS) and set(TCR_P1_ROWS+TCR_P2_ROWS)==set(TABLE_SCHEMAS["TCR"]["postes"]))
check("TCR_FULL expose toutes les lignes",rows_for_template("TCR_FULL")==TCR_P1_ROWS+TCR_P2_ROWS)
_t, _spec, _short, _ = compact_contract("TCR_FULL")
check("TCR_FULL garde exactement 4 colonnes",_t=="TCR" and _spec["cols"]==TCR_COLUMNS)
check("TCR_FULL ids sans doublon",len(_short)==len(set(_short.values()))==len(TABLE_SCHEMAS["TCR"]["postes"]))
check("Prompt TCR_FULL construit","TCR_FULL" in PROMPTS_EXTRACT and "N DEBIT" in PROMPTS_EXTRACT["TCR_FULL"] and "N-1 CREDIT" in PROMPTS_EXTRACT["TCR_FULL"])

# Cas de routage critiques
check("A09 n'est jamais TCR",route_from_text("9/ Tableau de détermination du résultat fiscal I. Résultat net de l'exercice II. Réintégrations III. Déductions Résultat fiscal")=="A09")
check("Annexe 3/4 n'est jamais TCR2",route_from_text("3/ Charges de personnel, impôts, taxes 4/ Autres charges et produits opérationnels Autres produits opérationnels")=="A03_A04")
check("TCR P2 sans entête reconnu",route_from_text("Autres produits opérationnels V-Résultat opérationnel VI-Résultat financier VII-Résultat ordinaire VIII-Résultat extraordinaire IX - RESULTAT NET DE L’EXERCICE")=="TCR_P2")
check("TCR P1 deux-pages reconnu",route_from_text("COMPTE DE RESULTAT Rubriques N N-1 DEBIT CREDIT Ventes de marchandises I-Production de l'exercice II-Consommations de l'exercice III-Valeur ajoutée IV-Excédent brut d'exploitation")=="TCR_P1")
_tcr_full_text="""COMPTE DE RESULTAT Rubriques N N-1 DEBIT CREDIT Ventes de marchandises I-Production de l'exercice II-Consommations de l'exercice III-Valeur ajoutée IV-Excédent brut d'exploitation Autres produits opérationnels V-Résultat opérationnel VI-Résultat financier VII-Résultat ordinaire VIII-Résultat extraordinaire IX-Résultat net de l'exercice"""
check("TCR complet une-page reconnu",route_from_text(_tcr_full_text)=="TCR_FULL",route_from_text(_tcr_full_text))
check("PASSIF structure officielle",route_from_text("BILAN (PASSIF) CAPITAUX PROPRES PASSIFS NON-COURANTS PASSIFS COURANTS TOTAL PASSIF (I+II+III)")=="PASSIF")

# Fusion logique : deux pages et une page doivent produire le même tableau TCR canonique.
_doc2={"pages":[
    {"numero_page_scannee":3,"classification":{"types":["TCR"]},"donnees":{"TCR":{"production_exercice":{"n_debit":1.0}}}},
    {"numero_page_scannee":4,"classification":{"types":["TCR"]},"donnees":{"TCR":{"resultat_net_exercice":{"n_credit":2.0}}}},
]}
_rows2,_pages2=table_rows(_doc2,"TCR")
_doc1={"pages":[
    {"numero_page_scannee":3,"classification":{"types":["TCR"]},"donnees":{"TCR":{"production_exercice":{"n_debit":1.0},"resultat_net_exercice":{"n_credit":2.0}}}},
]}
_rows1,_pages1=table_rows(_doc1,"TCR")
check("TCR 2 pages fusionné",set(_rows2)=={"production_exercice","resultat_net_exercice"} and _pages2==[3,4])
check("TCR 1 page fusionné",set(_rows1)=={"production_exercice","resultat_net_exercice"} and _pages1==[3])
check("TCR canonique identique 1p/2p",_rows1==_rows2)

# Seuils d'écart demandés
check("999 DZD = OK",ecart_level(999)=="OK")
check("1000 DZD = LOW",ecart_level(1000)=="LOW")
check("10000 DZD = MEDIUM",ecart_level(10000)=="MEDIUM")
check("100000 DZD = HIGH",ecart_level(100000)=="HIGH")


# Identité client / seuil 92 %
check("Similarité exacte = 100%",abs((company_similarity("AGRANA SPA","AGRANA S.P.A") or 0)-1.0)<1e-9)
check("Seuil identité 92%",IDENTITY_SIMILARITY_MIN==0.92)
_ref={"year":2025,"entreprise":"AGRANA ALGERIE SPA","nif_15":"123456789012345"}
_doc_ok={"document":{"fichier_source":"2024.pdf"},"pages":[{"numero_page_scannee":2,"classification":{"types":["PASSIF"]},"entete_page":{"entreprise":"AGRANA ALGERIE S.P.A","nif_15":"123456789012345"},"donnees":{"PASSIF":{"capital_emis":{"n":1}}}}]}
_aok=audit_table_identity(_doc_ok,"PASSIF",2024,_ref)
check("Identité comparable conforme",_aok["level"]=="OK",repr(_aok))
_doc_bad={"document":{"fichier_source":"2024.pdf"},"pages":[{"numero_page_scannee":2,"classification":{"types":["PASSIF"]},"entete_page":{"entreprise":"AUTRE SOCIETE SPA","nif_15":"999999999999999"},"donnees":{"PASSIF":{"capital_emis":{"n":1}}}}]}
_abad=audit_table_identity(_doc_bad,"PASSIF",2024,_ref)
check("NIF/raison sociale différents = HIGH",_abad["level"]=="HIGH",repr(_abad))
_doc_tcr_mix={"document":{"fichier_source":"2024.pdf"},"pages":[
 {"numero_page_scannee":3,"classification":{"types":["TCR"]},"entete_page":{"entreprise":"AGRANA ALGERIE SPA","nif_15":"123456789012345"},"donnees":{"TCR":{"production_exercice":{"n_credit":1}}}},
 {"numero_page_scannee":4,"classification":{"types":["TCR"]},"entete_page":{"entreprise":"AUTRE SOCIETE","nif_15":"999999999999999"},"donnees":{"TCR":{"resultat_net_exercice":{"n_credit":1}}}}]}
_atmix=audit_table_identity(_doc_tcr_mix,"TCR",2024,_ref)
check("TCR 2 pages de clients différents = HIGH",_atmix["level"]=="HIGH" and _atmix["internal_nif_bad"],repr(_atmix))


# Validation du formulaire de référence à deux pages TCR s'il est présent dans le runtime.
ref=Path("/mnt/data/GN2_administration_uniquement.pdf")
if ref.exists():
    expected=["ACTIF","PASSIF","TCR_P1","TCR_P2","A01_A02","A03_A04","A05_A06","A07_A08","A08_1_A08_2","A09","A10_A11","A12_A13"]
    got=[route_from_text(p["text"]) for p in pdf_manifest(ref)]
    check("12 pages formulaire G2 routées exactement",got==expected,"got="+repr(got))

# Vérification du modèle Excel sans le modifier.
if MODEL_XLSX.exists():
    wbtest=load_workbook(MODEL_XLSX,read_only=False,data_only=False)
    check("Feuilles Excel attendues",all(s in wbtest.sheetnames for s in [FEUILLE_ACTIF,FEUILLE_PASSIF,FEUILLE_TCR]))
    check("PASSIF total N est formule",is_formula(wbtest[FEUILLE_PASSIF]["H28"]))
    check("TCR résultat net N Débit est formule",is_formula(wbtest[FEUILLE_TCR]["M55"]))
    check("TCR résultat net N Crédit est formule",is_formula(wbtest[FEUILLE_TCR]["N55"]))


# V26 — contrôle exhaustif des formules financières + présentation sans remplissage
check("Seuil formule LOW 1000 DZD",ecart_level(1000)=="LOW")
check("Police alerte formule rouge",FORMULA_ALERT_FONT_RGB=="FFFF0000")
check("Alias ACTIF en-tête total non courant",ACTIF_FORMULA_ROW_ALIASES.get(9)=="total_actif_non_courant")
check("Alias ACTIF en-tête total courant",ACTIF_FORMULA_ROW_ALIASES.get(25)=="total_actif_courant")
check("Alias PASSIF capitaux propres",PASSIF_FORMULA_ROW_ALIASES.get(5)=="total_capitaux_propres")
check("Alias PASSIF non courant",PASSIF_FORMULA_ROW_ALIASES.get(16)=="total_passifs_non_courants")
check("Alias PASSIF courant",PASSIF_FORMULA_ROW_ALIASES.get(22)=="total_passifs_courants")

if MODEL_XLSX.exists():
    # Document synthétique : toutes les rubriques contiennent des valeurs afin de tester l'inventaire des cellules formule.
    _dummy_rows={}
    for _t in ("ACTIF","PASSIF","TCR"):
        _dummy_rows[_t]={}
        for _rc in TABLE_SCHEMAS[_t]["postes"]:
            _dummy_rows[_t][_rc]={c:1000.0 for c in TABLE_SCHEMAS[_t]["cols"]}
    _dummy={"document":{"fichier_source":"dummy_2025.pdf"},"identite":{"annee":2025},"pages":[
        {"numero_page_scannee":1,"classification":{"types":["ACTIF"]},"donnees":{"ACTIF":_dummy_rows["ACTIF"]}},
        {"numero_page_scannee":2,"classification":{"types":["PASSIF"]},"donnees":{"PASSIF":_dummy_rows["PASSIF"]}},
        {"numero_page_scannee":3,"classification":{"types":["TCR"]},"donnees":{"TCR":_dummy_rows["TCR"]}},
    ]}
    _cands={(2025,t):(0,_dummy,"N") for t in ("ACTIF","PASSIF","TCR")}
    _wb=load_workbook(MODEL_XLSX,read_only=False,data_only=False)
    _fc=build_all_financial_formula_controls(_wb,_cands,{2025:"N"})
    _coords={(x["sheet"],x["cell"]) for x in _fc}
    check("73 formules financières comparables par exercice",len(_fc)==73,f"count={len(_fc)}")
    check("ACTIF formule Net terrain contrôlée",(FEUILLE_ACTIF,"R13") in _coords)
    check("ACTIF total général contrôlé",(FEUILLE_ACTIF,"R36") in _coords)
    check("PASSIF formule de tête capitaux propres contrôlée",(FEUILLE_PASSIF,"H5") in _coords)
    check("PASSIF total général contrôlé",(FEUILLE_PASSIF,"H28") in _coords)
    check("PASSIF contrôles techniques 29/30 exclus",(FEUILLE_PASSIF,"H29") not in _coords and (FEUILLE_PASSIF,"H30") not in _coords)
    check("TCR chiffre affaires Débit+Crédit contrôlés",(FEUILLE_TCR,"M12") in _coords and (FEUILLE_TCR,"N12") in _coords)
    check("TCR résultat net Débit+Crédit contrôlés",(FEUILLE_TCR,"M55") in _coords and (FEUILLE_TCR,"N55") in _coords)
    _tmp=openpyxl.Workbook(); _ws=_tmp.active; _ws.title="TEST"; _c=_ws["A1"]; _c.value="=1+1"; _c.fill=PatternFill("solid",start_color="D9EAF7")
    _before=fill_signature(_c)
    apply_formula_alerts(_tmp,[{"sheet":"TEST","cell":"A1","level":"HIGH","status":"ECART_FORMULE","table":"TCR","row":"resultat_net_exercice","column":"n_credit","year":2025,"observed_kdzd":100.0,"calculated_kdzd":90.0,"difference_dzd":-10000.0,"formula":"=1+1","source":"dummy.pdf — p.3"}])
    check("Alerte formule ajoute commentaire",_c.comment is not None)
    check("Alerte formule met chiffre en rouge",font_is_formula_alert_red(_c))
    check("Alerte formule ne change pas le fond",fill_signature(_c)==_before)

print(f"\nTESTS V26: {sum(_tests)}/{len(_tests)} réussis")
if not all(_tests): raise AssertionError("Au moins un test V26 a échoué — ne pas industrialiser avant correction")


## Sorties V26

- JSON bruts : `/mnt/Risk/bilans_out/GT/V26/json_bruts_v25`
- JSON avant certification : `/mnt/Risk/bilans_out/GT/V26/json_avant_certification_v25`
- Excel pré-certification : `/mnt/Risk/bilans_out/GT/V26/excel_pre_certification_v25`

Pour bénéficier du contrôle d'identité **page par page**, V26 doit retraiter les PDF au moins une fois : les anciens JSON V24b ne contiennent pas nécessairement l'identité des pages non extraites (annexes).
